# Colorado 14er route ingestion — Block 1

This notebook embeds the complete scraper and the exact 100-entry requested manifest. Read **project_roadmap.md** first: the updated 100-selection manifest includes three approved replacements, Class 5 = 16, and retained related itineraries. Live downloads were blocked with HTTP 403 in the development environment; HTML/download selectors remain unverified.

The pipeline reports missing/ambiguous routes rather than substituting tracks. Obtain source permission for your intended automated/derived/public use and review the source download agreement before enabling execution. It stops on robots denial, authentication errors and site access challenges.

In [ ]:
%pip install -q requests beautifulsoup4 pandas defusedxml numpy


## Optional persistent storage

Colab's `/content` is temporary. Run the following cell only if you want a persistent private Google Drive folder. Keep cookies, cached source pages and GPX out of public repositories.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/14er_research_private'
else:
    DATA_ROOT = '/content/14er_research_private'
print(DATA_ROOT)

## Complete implementation

Definitions only: no network request is made by this cell. The same code is available as `block1_scraper.py`.

In [ ]:
"""Colorado 14er ingestion, Python 3.10+. No endpoint guessing or fuzzy auto-matches.

Install: pip install requests beautifulsoup4 pandas defusedxml
See project_roadmap.md and the accompanying Colab notebook for execution.
"""
from __future__ import annotations

import argparse
import csv
import difflib
import hashlib
import html
import importlib.metadata
import json
import math
import os
import random
import re
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from http.cookiejar import MozillaCookieJar
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urljoin, urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from defusedxml import ElementTree as ET

BASE = 'https://www.14ers.com/'
BOT = 'Colorado14erResearch'
MAX_BYTES = 20_000_000
RISK_LABELS = ('Low', 'Moderate', 'Considerable', 'High', 'Extreme')
YDS_ENCODING = {1: 1, 2: 2, 3: 4, 4: 8, 5: 16}


class AccessBlocked(RuntimeError):
    pass


class IngestionError(RuntimeError):
    pass


def now():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)


def atomic_bytes(path, body):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_bytes(body)
    temp.replace(path)


def canonical(url):
    """Only follow same-site HTTPS URLs. Never send cookies to another host."""
    p = urlsplit(html.unescape(urljoin(BASE, url)))
    if p.scheme != 'https' or p.hostname not in ('14ers.com', 'www.14ers.com'):
        raise IngestionError('Unexpected host or non-HTTPS URL')
    if p.username or p.password:
        raise IngestionError('Credentials in URL are not supported')
    if p.port not in (None, 443):
        raise IngestionError('Unexpected port')
    return urlunsplit(('https', 'www.14ers.com', p.path or '/', p.query, ''))


def norm(text):
    text = text.casefold().replace('mt.', 'mount').replace('mt ', 'mount ')
    text = text.replace('mount of the holy cross', 'holy cross')
    text = text.replace('mount evans', 'mount blue sky')
    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'\bstandard route\b', ' ', text)
    text = re.sub(r'\bthe\b', ' ', text)
    return ' '.join(re.findall(r'[a-z0-9]+', text))


def explicit_urls(soup, base_url):
    """Extract literal links, including inline JS strings, without executing JS.

    Function-generated URLs are intentionally unresolved, never synthesized.
    """
    values = []
    for node in soup.find_all(True):
        for attr in ('href', 'data-href', 'data-url', 'data-download', 'action'):
            value = node.get(attr)
            if isinstance(value, str):
                values.append(value)
        handler = node.get('onclick', '')
        values.extend(re.findall(r'''["']([^"']+)["']''', handler))
    for script in soup.find_all('script'):
        values.extend(re.findall(r'''["']([^"'\r\n]+)["']''', script.get_text()))
    result = set()
    for value in values:
        value = value.replace('\\/', '/')
        if value.startswith(('javascript:', '#', 'mailto:')):
            continue
        if not re.search(r'(\.php(?:\?|$)|\.gpx(?:\?|$))', value, re.I):
            continue
        if any(c in value for c in (' ', '{', '}', '<', '>')):
            continue
        try:
            result.add(canonical(urljoin(base_url, value)))
        except IngestionError:
            continue
    return sorted(result)


def route_url(url):
    p = urlsplit(url)
    q = parse_qs(p.query)
    if p.path.endswith('/route.php') and len(q.get('route', [])) == 1:
        rid = q['route'][0]
        if re.fullmatch(r'[a-zA-Z0-9_-]+', rid):
            return BASE + 'route.php?' + urlencode({'route': rid})
    return None


class Client:
    def __init__(self, root, contact, cookie_file=None, delay=4.0):
        self.root = Path(root)
        self.cache = self.root / 'cache'
        self.cache.mkdir(parents=True, exist_ok=True)
        self.log = self.root / 'events.jsonl'
        self.session = requests.Session()
        self.session.headers.update({'User-Agent': f'{BOT}/0.1 ({contact})'})
        if cookie_file:
            jar = MozillaCookieJar(str(cookie_file))
            jar.load(ignore_discard=True, ignore_expires=False)
            for cookie in jar:
                if cookie.domain.lstrip('.') in ('14ers.com', 'www.14ers.com'):
                    self.session.cookies.set_cookie(cookie)
        self.delay = max(3.0, delay)
        self.last_request = 0.0
        self.robots = None

    def event(self, **entry):
        # No headers, cookies, passwords, or response bodies in the log.
        with self.log.open('a', encoding='utf-8') as f:
            f.write(json.dumps({'time': now(), **entry}, ensure_ascii=False) + '\n')

    def wait(self):
        remaining = self.delay + random.uniform(0, 1) - (time.monotonic() - self.last_request)
        if remaining > 0:
            time.sleep(remaining)
        self.last_request = time.monotonic()

    def check_robots(self):
        """Fail closed if robots cannot be retrieved, except HTTP 404."""
        self.wait()
        try:
            response = self.session.get(BASE + 'robots.txt', timeout=(10, 45),
                                        allow_redirects=False)
        except requests.RequestException as exc:
            raise AccessBlocked('robots.txt unavailable; no crawl started') from exc
        self.robots = RobotFileParser()
        if response.status_code == 404:
            self.robots.parse(['User-agent: *', 'Allow: /'])
        elif response.status_code == 200:
            body = response.text
            if '<html' in body.lower() or 'captcha' in body.lower():
                raise AccessBlocked('robots.txt returned HTML/challenge')
            self.robots.parse(body.splitlines())
        else:
            raise AccessBlocked(f'robots.txt HTTP {response.status_code}; no crawl started')
        for value in (self.robots.crawl_delay(BOT), self.robots.crawl_delay('*')):
            if value:
                self.delay = max(self.delay, float(value))
        rate = self.robots.request_rate(BOT) or self.robots.request_rate('*')
        if rate and rate.requests > 0:
            # Serialize requests rather than burst at the allowed rate.
            self.delay = max(self.delay, rate.seconds / rate.requests)

    def fetch(self, url, refresh=False, referer=None):
        url = canonical(url)
        if self.robots is None:
            raise IngestionError('Call check_robots() before fetch()')
        if not self.robots.can_fetch(BOT, url):
            raise IngestionError('robots.txt disallows this URL')
        key = hashlib.sha256(url.encode()).hexdigest()
        path = self.cache / (key + '.bin')
        if path.exists() and not refresh:
            sidecar = self.cache / (key + '.json')
            cached = path.read_bytes()
            metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            if metadata.get('url') == url and metadata.get('sha256') == hashlib.sha256(cached).hexdigest():
                return cached
            self.event(kind='cache_integrity_mismatch', url=url)
        current = url
        redirects = 0
        attempts = 0
        while attempts < 5:
            if not self.robots.can_fetch(BOT, current):
                raise IngestionError('robots.txt disallows redirect destination')
            self.wait()
            try:
                with self.session.get(current, timeout=(10, 60), allow_redirects=False,
                                      stream=True,
                                      headers={'Referer': referer} if referer else {}) as r:
                    status = r.status_code
                    self.event(kind='request', url=current, status=status)
                    if status in (401, 403):
                        raise AccessBlocked(f'HTTP {status}; stop and resolve site access')
                    if status in (301, 302, 303, 307, 308):
                        redirects += 1
                        if redirects > 5 or not r.headers.get('Location'):
                            raise IngestionError('Invalid redirect chain')
                        current = canonical(urljoin(current, r.headers['Location']))
                        if re.search(r'(login|signin)', urlsplit(current).path, re.I):
                            raise AccessBlocked('Login required or session expired')
                        continue
                    if status in (429, 500, 502, 503, 504):
                        attempts += 1
                        wait = 2 ** attempts * 5 + random.uniform(0, 2)
                        retry_after = r.headers.get('Retry-After')
                        if retry_after:
                            try:
                                wait = max(wait, float(retry_after))
                            except ValueError:
                                try:
                                    wait = max(wait, parsedate_to_datetime(retry_after).timestamp() - time.time())
                                except (ValueError, TypeError, OverflowError):
                                    pass
                        self.event(kind='backoff', url=current, seconds=wait)
                        # A very long server cooldown is honored by ending this run.
                        if wait > 300:
                            raise AccessBlocked(f'Server cooldown {wait:.0f}s; rerun later')
                        time.sleep(wait)
                        continue
                    if status != 200:
                        raise IngestionError(f'HTTP {status}')
                    chunks, size = [], 0
                    for chunk in r.iter_content(65536):
                        size += len(chunk)
                        if size > MAX_BYTES:
                            raise IngestionError('Response exceeds 20 MB cap')
                        chunks.append(chunk)
                    body = b''.join(chunks)
                    prefix = body[:300_000].decode('utf-8', errors='ignore').lower()
                    if any(marker in prefix for marker in (
                        'cf-chl-', 'challenge-platform', 'verify you are human',
                        'checking your browser', 'access denied', 'g-recaptcha')):
                        raise AccessBlocked('Access challenge; no bypass attempted')
                    # Ordinary map verification widgets do not imply the whole
                    # route page is blocked; the metadata/GPX controls may work.
                    if b'<html' in body.lower() and re.search(
                        r'<input[^>]+type\s*=\s*["\']password', prefix):
                        raise AccessBlocked('Login form returned; authorized session required')
                    atomic_bytes(path, body)
                    write_json(self.cache / (key + '.json'), {
                        'url': url, 'final_url': current, 'fetched_at': now(),
                        'sha256': hashlib.sha256(body).hexdigest()})
                    return body
            except (requests.Timeout, requests.ConnectionError):
                attempts += 1
                self.event(kind='network_retry', url=current, attempt=attempts)
                time.sleep(2 ** attempts * 5)
        raise IngestionError('Retry budget exhausted')


def metric_candidates(overview, label, unit):
    """Keep every published total and its label; ambiguity is never first-number wins."""
    section = re.search(label + r'\s+(.*?)(?=Elevation\s+Gain|Length|Difficulty|Exposure|Rockfall|Route[ -]Finding|Commitment|Peak\s+Conditions|Downloads|Route\s+Last\s+Updated|\bGPX\b|\bKML\b|$)', overview, re.I)
    if not section:
        return []
    text = section.group(1)
    # Actual overviews use both prefix labels and suffix labels. A shared unit
    # may follow only the last prefix total (Harvard-Columbia's distance).
    if re.match(r'(?:From|With)\s+', text, re.I) and re.search(unit, text, re.I):
        return [{'value': float(m.group(2).replace(',', '')), 'start_label': m.group(1).strip()}
                for m in re.finditer(r'(?:From|With)\s+([^:]+):\s*([\d,]+(?:\.\d+)?)', text, re.I)]
    matches = list(re.finditer(r'([\d,]+(?:\.\d+)?)\s*' + unit, text, re.I))
    candidates = []
    for i, match in enumerate(matches):
        tail = text[match.end():matches[i+1].start() if i+1 < len(matches) else len(text)]
        tail = tail.strip(' :;,()-')
        label_text = re.sub(r'^(?:from|starting at|starting near|if you start at|if you start near)\s+', '', tail, flags=re.I)
        # Explanatory notes are not a start location. Preserve them separately.
        if re.match(r'^(?:via|if you descend)\s+', tail, re.I):
            label_text = tail
        elif not re.match(r'^(?:from|starting at|starting near|if you start at|if you start near)\s+', tail, re.I):
            label_text = ''
        candidates.append({'value': float(match.group(1).replace(',', '')), 'start_label': label_text,
                           **({'scope_note': tail} if tail and not label_text else {})})
    return candidates


def start_norm(text):
    """Start labels retain drive/access qualifiers; parentheses are meaningful."""
    text = re.sub(r'\btrailhead\b', 'th', text.lower())
    return re.sub(r'[^a-z0-9]+', ' ', text).strip()


def apply_selection_metadata(row, source):
    result = dict(source)
    result['yds_observed_class'] = source.get('yds_class')
    result['yds_effective_provenance'] = source.get('source_url', '')
    override = row.get('yds_class_override', '')
    if override:
        if not row.get('yds_override_provenance') or int(override) not in YDS_ENCODING:
            raise IngestionError('Unsupported or unprovenanced YDS override')
        result['yds_class'] = int(override)
        result['yds_encoded'] = YDS_ENCODING[int(override)]
        result['yds_effective_provenance'] = row['yds_override_provenance']
    unresolved = []
    for field in ('source_distance_mi', 'source_gain_ft'):
        candidates = source.get(field + '_candidates', [])
        if len(candidates) > 1:
            labels = {start_norm(row.get('start_name', ''))}
            if row.get('descent_policy') in ('reverse_ascent', 'West Ridge to Blue Lakes; no Cristo Couloir'):
                labels.add(start_norm('if you descend the ascent route'))
            if row.get('source_url', '').endswith('route=oxfo2') and row.get('start_name') == 'Missouri Gulch':
                labels.add(start_norm('via Belford standard route'))
            selected = [c for c in candidates if start_norm(c['start_label']) in labels and c['start_label']]
            result[field] = selected[0]['value'] if len(selected) == 1 else None
            if len(selected) != 1:
                unresolved.append(field)
        elif candidates:
            label = candidates[0]['start_label']
            # A named single total must also match a named chosen start.
            if label and row.get('start_name') and start_norm(label) != start_norm(row['start_name']):
                result[field] = None
                unresolved.append(field)
    result['metadata_scope_status'] = 'review_required' if unresolved else 'single_or_matched_published_total'
    result['metadata_scope_unresolved_fields'] = unresolved
    return result


def parse_route(body, url):
    soup = BeautifulSoup(body, 'html.parser')
    h1 = soup.find('h1')
    if not h1:
        raise IngestionError('No route heading: HTML layout changed or access denied')
    title = h1.get_text(' ', strip=True)
    # Restrict numeric extraction to the overview, ahead of narrative directions.
    text = soup.get_text(' ', strip=True)
    start = text.find(title)
    overview = text[max(0, start):]
    end = re.search(r'Route Last Updated:', overview)
    if end:
        overview = overview[:end.start()]
    def extract(pattern):
        m = re.search(pattern, overview, re.I)
        return m.group(1) if m else None
    rating = extract(r'Difficulty\s+((?:Easy\s+|Difficult\s+)?Class\s+\d(?:\.\d+[a-d]?)?)')
    number = re.search(r'Class\s+(\d)', rating or '')
    cls = int(number.group(1)) if number else None
    gains = metric_candidates(overview, r'Elevation\s+Gain', r'(?:ft|feet|\')')
    distances = metric_candidates(overview, r'Length(?:\s+Round[ -]Trip)?(?:\s+RT)?', r'mi(?:les)?')
    risk = {}
    for field, label in [('exposure', r'Exposure'),
                         ('rockfall', r'Rockfall(?:\s+Potential)?'),
                         ('route_finding', r'Route[ -]Finding'),
                         ('commitment', r'Commitment')]:
        raw = extract(label + r'\s+(' + '|'.join(RISK_LABELS) + r')\b')
        risk[field + '_raw'] = raw
    gpx = set()
    for link in explicit_urls(soup, url):
        p = urlsplit(link)
        q = parse_qs(p.query)
        if p.path.lower().endswith('.gpx') or any(
            v.lower() == 'gpx' for values in q.values() for v in values):
            gpx.add(link)
    # GPX links may use an opaque PHP endpoint with GPX in control text/image alt.
    for node in soup.find_all(['a', 'button']):
        label = node.get_text(' ', strip=True) + ' ' + ' '.join(
            img.get('alt', '') for img in node.find_all('img'))
        if re.search(r'\bgpx\b', label, re.I):
            miniature = BeautifulSoup(str(node), 'html.parser')
            gpx.update(link for link in explicit_urls(miniature, url)
                       if not urlsplit(link).path.lower().endswith(('.png', '.jpg')))
    gpx.discard(url)
    snow_only = bool(re.search(r'snow[ -]only (?:climb|route)', title + ' ' + overview, re.I))
    snow_only = snow_only or any('snow-only' in str(img.get('alt', '')).lower()
                                 for img in soup.find_all('img'))
    return {'source_url': url, 'source_title': title,
            'source_html_sha256': hashlib.sha256(body).hexdigest(),
            'yds_raw': rating, 'yds_class': cls,
            'yds_encoded': YDS_ENCODING.get(cls),
            'source_distance_mi': distances[0]['value'] if len(distances) == 1 else None,
            'source_gain_ft': gains[0]['value'] if len(gains) == 1 else None,
            'source_distance_mi_candidates': distances,
            'source_gain_ft_candidates': gains,
            'snow_only': snow_only, 'gpx_candidates': sorted(gpx), **risk}


def validate_gpx(body):
    """Validate GPX geometry; don't treat waypoints as a hike or join segments."""
    try:
        root = ET.fromstring(body)
    except Exception as exc:
        raise IngestionError('Download is not safe, parseable GPX XML') from exc
    def tag(node):
        return node.tag.rsplit('}', 1)[-1]
    if tag(root) != 'gpx':
        raise IngestionError('XML root is not gpx (possibly HTML login response)')
    sequences = [node for node in root.iter() if tag(node) in ('trkseg', 'rte')]
    points = []
    signature = []
    sequences_used = 0
    with_elevation = 0
    zero_steps = 0
    max_step_m = 0.0
    horizontal_m = 0.0
    endpoints = []
    for sequence in sequences:
        seq = []
        for node in sequence:
            if tag(node) not in ('trkpt', 'rtept'):
                continue
            try:
                lat, lon = float(node.attrib['lat']), float(node.attrib['lon'])
                if not (math.isfinite(lat) and math.isfinite(lon)
                        and -90 <= lat <= 90 and -180 <= lon <= 180):
                    raise ValueError('invalid coordinate')
                elevation = next((float(child.text) for child in node
                                  if tag(child) == 'ele'), None)
                if elevation is not None and not math.isfinite(elevation):
                    raise ValueError('invalid elevation')
            except (KeyError, ValueError, TypeError) as exc:
                raise IngestionError('Invalid GPX coordinates/elevation') from exc
            seq.append((lat, lon, elevation))
        if len(seq) >= 2:
            sequences_used += 1
            endpoints.append({"start": seq[0], "finish": seq[-1]})
            for a, b in zip(seq, seq[1:]):
                p1, p2 = math.radians(a[0]), math.radians(b[0])
                dp, dl = p2-p1, math.radians(b[1]-a[1])
                hav = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
                d = 6_371_008.8 * 2 * math.asin(math.sqrt(min(1, max(0, hav))))
                zero_steps += d < 0.01
                max_step_m = max(max_step_m, d)
                horizontal_m += d
            points.extend(seq)
            # Excludes timestamps and preserves sequence boundaries/order.
            signature.append([(round(a, 6), round(b, 6),
                               round(c, 1) if c is not None else None) for a,b,c in seq])
            with_elevation += sum(c is not None for _,_,c in seq)
    if not points:
        raise IngestionError('GPX has no track/route sequence with at least 2 points')
    # Broad regional check, not proof of route identity.
    if not all(36.8 <= a <= 41.2 and -109.2 <= b <= -101.8 for a,b,_ in points):
        raise IngestionError('Track falls outside broad Colorado bounds')
    return {'point_count': len(points), 'sequence_count': sequences_used,
            'elevation_coverage': with_elevation / len(points),
            'zero_length_steps': zero_steps, 'max_horizontal_step_m': max_step_m,
            'track_horizontal_distance_mi': horizontal_m / 1609.344,
            'sequence_endpoints': endpoints,
            'geometry_sha256': hashlib.sha256(json.dumps(signature).encode()).hexdigest(),
            'gpx_sha256': hashlib.sha256(body).hexdigest()}


def load_manifest(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    if len(rows) != 100 or len({r['route_id'] for r in rows}) != 100:
        raise ValueError('Manifest must contain exactly 100 unique route IDs')
    if {int(r['selection_number']) for r in rows} != set(range(1, 101)):
        raise ValueError('Selection numbers must be exactly 1..100')
    return rows


def discover(client, refresh=False):
    """Index -> advertised peak route lists -> advertised route detail URLs."""
    body = client.fetch(BASE + 'routes.php', refresh=refresh)
    soup = BeautifulSoup(body, 'html.parser')
    urls = explicit_urls(soup, BASE + 'routes.php')
    peak_lists = sorted({u for u in urls if urlsplit(u).path.endswith('/routelist.php')
                         and 'peakid' in parse_qs(urlsplit(u).query)})
    route_urls = {route_url(u) for u in urls if route_url(u)}
    if not peak_lists and not route_urls:
        raise IngestionError('Route index exposes no route links; inspect cached HTML')
    for url in peak_lists:
        try:
            page = BeautifulSoup(client.fetch(url, refresh=refresh), 'html.parser')
            route_urls.update(route_url(u) for u in explicit_urls(page, url) if route_url(u))
        except AccessBlocked:
            raise
        except (IngestionError, requests.RequestException) as exc:
            client.event(kind='discovery_error', url=url, message=str(exc))
    catalog = []
    for url in sorted(route_urls):
        try:
            catalog.append(parse_route(client.fetch(url, refresh=refresh), url))
        except AccessBlocked:
            # Save already discovered metadata before stopping the entire site crawl.
            write_json(client.root / 'catalog.partial.json', catalog)
            raise
        except (IngestionError, requests.RequestException) as exc:
            client.event(kind='catalog_error', url=url, message=str(exc))
    write_json(client.root / 'catalog.json', catalog)
    return catalog


def choose(row, catalog):
    """Source URL overrides are explicit. Otherwise only exact normalized names."""
    if row.get('source_url', '').strip():
        url = route_url(canonical(row['source_url']))
        if not url:
            raise IngestionError('source_url must be an explicit route.php?route=... URL')
        entries = [r for r in catalog if r['source_url'] == url]
    else:
        names = [row.get('canonical_name', ''), row['requested_name']] + row.get('match_names', '').split('|')
        keys = {norm(n) for n in names if n.strip()}
        entries = [r for r in catalog if norm(r['source_title']) in keys]
    if len(entries) == 1:
        return entries[0], []
    options = entries or sorted(catalog, key=lambda r: difflib.SequenceMatcher(
        None, norm(row['requested_name']), norm(r['source_title'])).ratio(), reverse=True)[:5]
    candidates = [{'title': r['source_title'], 'url': r['source_url']} for r in options]
    return None, candidates


def load_curated(path, manifest):
    """Validate an owner-approved local export without issuing network requests."""
    document = json.loads(Path(path).read_text(encoding='utf-8'))
    digest = hashlib.sha256(Path(manifest).read_bytes()).hexdigest()
    if document.get('manifest_sha256') != digest or document.get('schema_version') != '1.0.0':
        raise IngestionError('Curated export schema/frozen manifest mismatch')
    if not isinstance(document.get('records'), list) or not document.get('permission_evidence'):
        raise IngestionError('Curated export requires records and permission evidence')
    records = {}
    ids = {r['route_id'] for r in load_manifest(manifest)}
    for record in document['records']:
        if not isinstance(record, dict):
            raise IngestionError('Curated records must be JSON objects')
        rid = record.get('route_id')
        if rid not in ids or rid in records:
            raise IngestionError('Unknown or duplicate curated route ID')
        records[rid] = record
    return document, records


def ingest_curated(row, record, document, export_path, root):
    """A curation adapter, not a trip-report scraping or inferred geometry adapter."""
    required = ('reviewer', 'review_date', 'metadata_evidence', 'geometry_evidence',
                'source_references', 'start_name', 'summit_order', 'descent_policy',
                'start_policy', 'yds_raw', 'gpx_file', 'gpx_sha256')
    if any(not record.get(k) for k in required):
        raise IngestionError('Curated record lacks required review/provenance fields')
    datetime.strptime(record['review_date'], '%Y-%m-%d')
    if not isinstance(record['source_references'], list) or any(
            not isinstance(u, str) or urlsplit(u).scheme != 'https' or not urlsplit(u).hostname
            for u in record['source_references']):
        raise IngestionError('Curated source references must be HTTPS URLs')
    for field in ('summit_order', 'descent_policy', 'start_policy'):
        if record[field] != row[field]:
            raise IngestionError('Curated itinerary differs from frozen manifest: ' + field)
    if row.get('start_name') and norm(row['start_name']) != norm(record['start_name']):
        raise IngestionError('Curated start differs from frozen manifest')
    for field in ('itinerary_verified', 'dry_summer_verified'):
        if type(record.get(field)) is not bool:
            raise IngestionError('Curated verification fields must be JSON booleans')
    if record['dry_summer_verified'] and not record.get('conditions_evidence'):
        raise IngestionError('Dry-summer verification requires conditions evidence')
    cls = record.get('yds_class')
    if type(cls) is not int or cls not in YDS_ENCODING:
        raise IngestionError('Invalid curated YDS class')
    raw_class = re.search(r'Class\s+([1-5])\b', record['yds_raw'], re.I)
    if not raw_class or int(raw_class.group(1)) != cls:
        raise IngestionError('Curated raw YDS and observed class disagree; use an explicit override')
    values = {}
    for field in ('source_distance_mi', 'source_gain_ft'):
        value = record.get(field)
        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
            raise IngestionError('Invalid positive curated metric: ' + field)
        values[field] = value
    for field in ('exposure_raw', 'rockfall_raw', 'route_finding_raw', 'commitment_raw'):
        if record.get(field) not in RISK_LABELS:
            raise IngestionError('Unknown or missing curated risk: ' + field)
        values[field] = record[field]
    base = Path(export_path).resolve().parent
    relative = Path(record['gpx_file'])
    source_path = (base / relative).resolve()
    if relative.is_absolute() or base not in source_path.parents:
        raise IngestionError('Curated GPX must be inside its export directory')
    if source_path.stat().st_size > MAX_BYTES:
        raise IngestionError('Curated GPX exceeds size cap')
    body = source_path.read_bytes()
    if hashlib.sha256(body).hexdigest() != record['gpx_sha256']:
        raise IngestionError('Curated GPX checksum mismatch')
    quality = validate_gpx(body)
    source = {**values, 'yds_raw': record['yds_raw'], 'yds_class': cls,
              'yds_encoded': YDS_ENCODING[cls], 'snow_only': False,
              'source_url': row.get('source_url', '')}
    row.update(apply_selection_metadata(row, source))
    path = Path(root) / 'gpx' / (row['route_id'] + '.gpx')
    atomic_bytes(path, body)
    provenance = {k: record[k] for k in required if k != 'gpx_file'}
    provenance.update(permission_evidence=document['permission_evidence'],
                      export_sha256=hashlib.sha256(Path(export_path).read_bytes()).hexdigest(),
                      validated_at=now(), **quality)
    write_json(path.with_suffix('.json'), provenance)
    row.update(quality, status='downloaded', gpx_path=str(path), metadata_complete=True,
               start_name=record['start_name'], ingestion_method='approved_local_export',
               curation_provenance=provenance,
               itinerary_verified=str(record['itinerary_verified']).lower(),
               dry_summer_verified=str(record['dry_summer_verified']).lower())


def export(root, results, catalog, run_error=None):
    """All 100 statuses are exported, even after a global access block."""
    root = Path(root)
    groups = defaultdict(list)
    for r in results:
        if r.get('geometry_sha256'):
            groups[r['geometry_sha256']].append(r['route_id'])
    duplicates = [ids for ids in groups.values() if len(ids) > 1]
    # Connected groups preserve transitive links between source aliases,
    # predeclared related itineraries and newly discovered shared geometry.
    parent = {r['route_id']: r['route_id'] for r in results}
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def join(ids):
        ids = [i for i in ids if i in parent]
        for i in ids[1:]:
            parent[find(i)] = find(ids[0])
    source_groups = defaultdict(list)
    for r in results:
        if r.get('source_url'):
            source_groups[r['source_url']].append(r['route_id'])
        join(r.get('related_itinerary_group', '').split('|'))
    for ids in list(groups.values()) + list(source_groups.values()):
        join(ids)
    connected = defaultdict(list)
    for i in parent:
        connected[find(i)].append(i)
    for r in results:
        others = groups.get(r.get('geometry_sha256'), [])
        r['duplicate_geometry_ids'] = '|'.join(i for i in others if i != r['route_id'])
        r['evaluation_group'] = '|'.join(sorted(connected[find(r['route_id'])]))
        r['training_eligible'] = bool(r['status'] == 'downloaded'
                                     and r.get('dry_summer_verified', '').lower() == 'true'
                                     and r.get('itinerary_verified', '').lower() == 'true'
                                     and not r.get('snow_only')
                                     and r.get('yds_encoded') is not None
                                     and r.get('elevation_coverage') == 1.0
                                     and r.get('metadata_complete'))
    # CSV stores candidate lists as JSON instead of Python repr strings.
    frame = pd.DataFrame([{k: json.dumps(v) if isinstance(v, (list,dict)) else v
                           for k,v in r.items()} for r in results])
    for name, subset in [('route_status.csv', frame),
                         ('missing_routes.csv', frame[frame.status != 'downloaded']),
                         ('review_required.csv', frame[(frame.status != 'downloaded') | ~frame.training_eligible |
                                                       (frame.duplicate_geometry_ids != '')])]:
        atomic_bytes(root / name, subset.to_csv(index=False).encode('utf-8'))
    statuses = frame.status.value_counts().to_dict()
    eligible = int(frame.training_eligible.sum())
    report = {'run_time': now(), 'requested': 100, 'catalog_routes': len(catalog),
              'statuses': statuses, 'downloaded': statuses.get('downloaded', 0),
              'training_eligible': eligible, 'duplicate_geometry_groups': duplicates,
              'complete_100_selections_dry_routes': eligible == 100,
              'complete_100_unique_dry_routes': eligible == 100 and not duplicates,
              'run_error': run_error}
    write_json(root / 'run_summary.json', report)
    return report


def ingest_browser_downloads(rows, download_dir, root):
    """Import normal UI downloads; acquisition never implies reviewed itinerary scope."""
    folder = Path(download_dir).resolve()
    ledger = json.loads((folder / 'download_ledger.json').read_text())
    records = {}
    for receipt in ledger:
        code = receipt.get('code', '')
        if not re.fullmatch(r'[a-z0-9_]+', code) or code in records:
            raise IngestionError('Invalid or duplicate browser receipt code')
        if receipt.get('agreement') != 'user_approved_displayed_terms_2026-09-30':
            raise IngestionError('Browser receipt lacks recorded user acceptance')
        if not receipt.get('downloaded_at') or not receipt.get('method', '').startswith('normal_browser_GPX'):
            raise IngestionError('Browser receipt lacks acquisition provenance')
        body = (folder / (code + '.gpx')).read_bytes()
        quality = validate_gpx(body)
        if quality['gpx_sha256'] != receipt.get('gpx_sha256'):
            raise IngestionError('Browser GPX receipt hash mismatch: ' + code)
        records[code] = receipt
    catalog = []
    sources = {}
    for row in rows:
        url = row.get('source_url')
        if not url or url in sources:
            continue
        code = parse_qs(urlsplit(url).query).get('route', [''])[0]
        receipt = records.get(code)
        if not receipt or canonical(receipt['url']) != canonical(url):
            continue
        body = (folder / (code + '_metadata.html')).read_bytes()
        if hashlib.sha256(body).hexdigest() != receipt.get('metadata_sha256'):
            raise IngestionError('Browser metadata receipt hash mismatch: ' + code)
        source = parse_route(body, url)
        sources[url] = (code, source)
        catalog.append(source)
    for row in rows:
        if row.get('status') != 'pending':
            continue
        if row.get('source_url') not in sources:
            row.update(status='awaiting_curation', error='No exact reviewed full-itinerary browser record')
            continue
        code, source = sources[row['source_url']]
        receipt = records[code]
        body = (folder / (code + '.gpx')).read_bytes()
        quality = validate_gpx(body)
        path = root / 'gpx' / (row['route_id'] + '.gpx')
        atomic_bytes(path, body)
        row.update(apply_selection_metadata(row, source), **quality,
                   ingestion_method='normal_browser_download_import', status='downloaded',
                   gpx_path=str(path), coverage_status='unreviewed_source_geometry',
                   coverage_note='Original source track; not certified as complete selected itinerary')
        endpoints = quality['sequence_endpoints']
        first, last = endpoints[0]['start'], endpoints[-1]['finish']
        lat1, lat2 = math.radians(first[0]), math.radians(last[0])
        h = math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(math.radians(last[1]-first[1])/2)**2
        row['endpoint_gap_m'] = 2 * 6371008.8 * math.asin(math.sqrt(min(1.0, h)))
        total = row.get('source_distance_mi')
        row['source_track_distance_ratio'] = quality['track_horizontal_distance_mi']/total if total else None
        # These are review priorities, not automatic coverage certifications.
        row['coverage_review_priority'] = (
            'shared_approach_and_summit_connector' if row['route_id'] == 'co14-058' else
            'approach_and_return' if row['start_policy'].startswith('Needleton') else
            'loop_or_traverse_return' if row['itinerary_type'] != 'out_and_back' else
            'start_and_same_path_return' if row['endpoint_gap_m'] > 500 else
            'closed_track_identity_and_scope')
        row['metadata_complete'] = all(row.get(f) is not None for f in (
            'yds_raw', 'source_distance_mi', 'source_gain_ft',
            'exposure_raw', 'rockfall_raw', 'route_finding_raw', 'commitment_raw'))
        write_json(path.with_suffix('.json'), {
            'route_id': row['route_id'], 'source_url': source['source_url'],
            'acquisition_receipt': receipt, 'source_html_sha256': source['source_html_sha256'],
            'validated_at': now(), 'coverage_status': row['coverage_status'],
            'endpoint_gap_m': row['endpoint_gap_m'],
            'source_track_distance_ratio': row['source_track_distance_ratio'],
            'coverage_review_priority': row['coverage_review_priority'], **quality})
    write_json(root / 'catalog.json', catalog)
    return catalog


def run(manifest, out, contact, source_permission_confirmed=False,
        download_agreement_accepted=False, cookie_file=None, refresh=False,
        curated_file=None, offline=False, only_routes=None, browser_download_dir=None):
    rows = load_manifest(manifest)
    root = Path(out)
    root.mkdir(parents=True, exist_ok=True)
    atomic_bytes(root / 'manifest_snapshot.csv', Path(manifest).read_bytes())
    write_json(root / 'run_provenance.json', {
        'started_at': now(), 'manifest_sha256': hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),
        'ingestion_version': '2.1.1', 'offline': offline,
        'browser_download_import': bool(browser_download_dir),
        'selected_routes': only_routes or [r['route_id'] for r in rows],
        'source_permission_confirmed': source_permission_confirmed,
        'download_agreement_accepted': download_agreement_accepted,
        'curated_export_supplied': bool(curated_file),
        'python_version': sys.version.split()[0],
        'dependencies': {name: importlib.metadata.version(name) for name in
                         ('requests', 'beautifulsoup4', 'pandas', 'defusedxml')},
    })
    results = [{**r, 'status': 'pending', 'error': '', 'candidates': [],
                'yds_observed_class': None, 'yds_class': None, 'yds_encoded': None} for r in rows]
    if only_routes and not set(only_routes).issubset({r['route_id'] for r in rows}):
        raise ValueError('Unknown smoke-test route ID')
    for r in results:
        if only_routes and r['route_id'] not in only_routes:
            r.update(status='not_selected', error='Outside requested smoke-test subset')
    catalog = []
    error = None
    if browser_download_dir:
        if curated_file or not offline:
            raise ValueError('Browser import requires offline mode and no curated export')
        catalog = ingest_browser_downloads(results, browser_download_dir, root)
        return export(root, results, catalog)
    if curated_file:
        try:
            document, records = load_curated(curated_file, manifest)
            for r in results:
                if r['status'] != 'pending' or r['route_id'] not in records:
                    continue
                try:
                    ingest_curated(r, records[r['route_id']], document, curated_file, root)
                except (IngestionError, ValueError, OSError, TypeError) as exc:
                    r.update(status='curation_error', error=str(exc))
        except (IngestionError, ValueError, OSError, TypeError) as exc:
            for r in results:
                if r['status'] == 'pending':
                    r.update(status='curation_error', error=str(exc))
            return export(root, results, catalog, str(exc))
    if offline:
        for r in results:
            if r['status'] == 'pending':
                r.update(status='awaiting_approved_export', error='No reviewed local export record supplied')
        return export(root, results, catalog)
    if not source_permission_confirmed or not download_agreement_accepted:
        for r in results:
            if r['status'] == 'pending':
                r.update(status='not_started', error='Confirm source permission and download agreement before execution')
        return export(root, results, catalog, 'Execution confirmations required')
    client = Client(root, contact, cookie_file=cookie_file)
    try:
        client.check_robots()
        pending = [r for r in results if r['status'] == 'pending']
        # Frozen mappings make a full 159-route source crawl unnecessary.
        if any(not r.get('source_url') and not r.get('source_review_status', '').endswith('requires_curation')
               and not r.get('source_review_status', '').endswith('requires_assembly') for r in pending):
            catalog = discover(client, refresh=refresh)
        # Explicit route mappings not present in the index are still usable.
        extra = {route_url(canonical(r['source_url'])) for r in pending if r.get('source_url')}
        for url in sorted(u for u in extra if u and u not in {c['source_url'] for c in catalog}):
            try:
                catalog.append(parse_route(client.fetch(url, refresh=refresh), url))
            except (IngestionError, ValueError, requests.RequestException) as exc:
                client.event(kind='catalog_error', url=url, message=str(exc))
        write_json(root / 'catalog.json', catalog)
        for r in results:
            if r['status'] != 'pending':
                continue
            try:
                if not r.get('source_url') and r.get('source_review_status', '').endswith(('requires_curation', 'requires_assembly')):
                    r.update(status='awaiting_curation', error='Reviewed full-itinerary metadata and local geometry required')
                    continue
                source, candidates = choose(r, catalog)
                r['candidates'] = candidates
                if source is None:
                    r.update(status='unresolved', error='No unique exact match; review candidates and update manifest')
                    continue
                r.update(apply_selection_metadata(r, source), ingestion_method='live_source')
                r['metadata_complete'] = all(r.get(f) is not None for f in (
                    'yds_raw', 'source_distance_mi', 'source_gain_ft',
                    'exposure_raw', 'rockfall_raw', 'route_finding_raw', 'commitment_raw'))
                if r['snow_only']:
                    r.update(status='excluded_snow_only', error='Incompatible with dry-summer scope')
                    continue
                links = [canonical(r['gpx_url'])] if r.get('gpx_url') else source['gpx_candidates']
                if len(links) != 1:
                    r.update(status='gpx_link_unresolved', error=f'{len(links)} GPX URLs found; explicit GPX mapping required')
                    continue
                url = links[0]
                path = root / 'gpx' / (r['route_id'] + '.gpx')
                previous = path.read_bytes() if path.exists() and not refresh else None
                # Only resume a local file if its sidecar proves URL provenance.
                sidecar = path.with_suffix('.json')
                prior = json.loads(sidecar.read_text()) if sidecar.exists() else {}
                valid_previous = (previous is not None and prior.get('gpx_url') == url
                                  and prior.get('gpx_sha256') == hashlib.sha256(previous).hexdigest())
                body = previous if valid_previous else client.fetch(
                    url, refresh=refresh, referer=source['source_url'])
                quality = validate_gpx(body)
                atomic_bytes(path, body)
                write_json(sidecar, {'route_id': r['route_id'], 'source_url': source['source_url'],
                                    'gpx_url': url, 'validated_at': now(), **quality})
                r.update(quality, status='downloaded', gpx_url=url, gpx_path=str(path))
                client.event(kind='downloaded', route_id=r['route_id'], sha256=quality['gpx_sha256'])
            except AccessBlocked:
                raise
            except (IngestionError, ValueError, OSError, requests.RequestException) as exc:
                r.update(status='error', error=str(exc))
                client.event(kind='route_error', route_id=r['route_id'], message=str(exc))
            finally:
                export(root, results, catalog)
    except (AccessBlocked, IngestionError, requests.RequestException, ValueError, OSError) as exc:
        error = str(exc)
        for r in results:
            if r['status'] == 'pending':
                r.update(status='blocked_or_not_attempted', error=error)
        client.event(kind='run_stopped', message=error)
    finally:
        report = export(root, results, catalog, error)
    return report



## Exact requested manifest

The embedded version 1.2.0 roster contains the 100 approved distinct official routes while preserving every original requested name. The canonical CSV is authoritative. This cell preserves an existing manifest so reruns do not erase reviewed mappings.

Official identities are mapped for all 100 distinct selections. Kiener's, Princeton Southwest Ridge, the custom Chicago Basin combination and six exact-source aliases have been replaced. Original requested labels remain as history; canonical names define the active cohort. A source URL must be an observed `route.php?route=...` URL; participant evidence and component URLs have their own fields.

Set `dry_summer_verified` and `itinerary_verified` only after reviewing complete geometry and conditions. Source identity verification alone does not establish either flag. Update the manifest version and reviewed hash in the project contract when changing the roster or scope.

In [ ]:
MANIFEST_CSV = 'route_id,selection_number,manifest_version,requested_name,canonical_name,selection_group,required_summit,summit_order,is_replacement,conditions_scope,source_url,supporting_source_url,source_component_urls,source_review_status,source_review_date,gpx_url,match_names,itinerary_type,start_policy,start_name,descent_policy,itinerary_verified,dry_summer_verified,route_scope_status,yds_class_override,yds_override_provenance,related_itinerary_group,review_note\nco14-001,1,1.2.0,Mt. Elbert - Northeast Ridge,Mt. Elbert - Northeast Ridge,standard_ranked_requested,Mt. Elbert,Mt. Elbert,false,dry_summer,https://www.14ers.com/route.php?route=elbe1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-002,2,1.2.0,Mt. Massive - East Slopes,Mt. Massive - East Slopes,standard_ranked_requested,Mt. Massive,Mt. Massive,false,dry_summer,https://www.14ers.com/route.php?route=mass1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-003,3,1.2.0,Mt. Harvard - South Slopes,Mt. Harvard - South Slopes,standard_ranked_requested,Mt. Harvard,Mt. Harvard,false,dry_summer,https://www.14ers.com/route.php?route=harv1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-004,4,1.2.0,Blanca Peak - Northwest Ridge,Blanca Peak - Northwest Ridge,standard_ranked_requested,Blanca Peak,Blanca Peak,false,dry_summer,https://www.14ers.com/route.php?route=blan1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,"8,000",reverse_ascent,false,false,published_summer_route,,,,\nco14-005,5,1.2.0,La Plata Peak - Northwest Ridge,La Plata Peak - Northwest Ridge,standard_ranked_requested,La Plata Peak,La Plata Peak,false,dry_summer,https://www.14ers.com/route.php?route=lapl1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-006,6,1.2.0,Uncompahgre Peak - South Ridge,Uncompahgre Peak - South Ridge,standard_ranked_requested,Uncompahgre Peak,Uncompahgre Peak,false,dry_summer,https://www.14ers.com/route.php?route=unco1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-007,7,1.2.0,Crestone Peak - South Face,Crestone Peak - South Face,standard_ranked_requested,Crestone Peak,Crestone Peak,false,dry_summer,https://www.14ers.com/route.php?route=cpea2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,SC 2WD TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-008,8,1.2.0,Mt. Lincoln - West Ridge,Mt. Lincoln - West Ridge,standard_ranked_requested,Mt. Lincoln,Mt. Lincoln,false,dry_summer,https://www.14ers.com/route.php?route=linc1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-009,9,1.2.0,Grays Peak - North Slopes,Grays Peak - North Slopes,standard_ranked_requested,Grays Peak,Grays Peak,false,dry_summer,https://www.14ers.com/route.php?route=gray1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-010,10,1.2.0,Mt. Antero - West Slopes,Mt. Antero - West Slopes,standard_ranked_requested,Mt. Antero,Mt. Antero,false,dry_summer,https://www.14ers.com/route.php?route=ante1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-011,11,1.2.0,Torreys Peak - South Slopes,Torreys Peak - South Slopes,standard_ranked_requested,Torreys Peak,Torreys Peak,false,dry_summer,https://www.14ers.com/route.php?route=torr2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-012,12,1.2.0,Castle Peak - Northeast Ridge,Castle Peak - Northeast Ridge,standard_ranked_requested,Castle Peak,Castle Peak,false,dry_summer,https://www.14ers.com/route.php?route=cast1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,the main trailhead,reverse_ascent,false,false,published_summer_route,,,,\nco14-013,13,1.2.0,Quandary Peak - East Ridge,Quandary Peak - East Ridge,standard_ranked_requested,Quandary Peak,Quandary Peak,false,dry_summer,https://www.14ers.com/route.php?route=quan1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-014,14,1.2.0,Mt. Blue Sky - West Ridge,Mt. Blue Sky - West Ridge from Summit Lake,standard_ranked_requested,Mt. Blue Sky,Mt. Blue Sky,false,dry_summer,https://www.14ers.com/route.php?route=evan2,,,official_identity_verified,2026-09-30,,,out_and_back,published_standard_start,Summit Lake,reverse_ascent,false,false,published_summer_route,,,,West Ridge baseline starts at Summit Lake; retain other Blue Sky approaches separately.\nco14-015,15,1.2.0,Longs Peak - Keyhole Route,Longs Peak - Keyhole Route,standard_ranked_requested,Longs Peak,Longs Peak,false,dry_summer,https://www.14ers.com/route.php?route=long1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Longs Peak Trailhead,Keyhole Route to Longs Peak Trailhead,false,false,published_summer_route,,,,\nco14-016,16,1.2.0,Mt. Wilson - Southwest Slopes,Mt. Wilson - Southwest Slopes,standard_ranked_requested,Mt. Wilson,Mt. Wilson,false,dry_summer,https://www.14ers.com/route.php?route=mwil6,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-017,17,1.2.0,Mt. Shavano - East Slopes,Mt. Shavano - East Slopes,standard_ranked_requested,Mt. Shavano,Mt. Shavano,false,dry_summer,https://www.14ers.com/route.php?route=shav1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-018,18,1.2.0,Mt. Princeton - East Slopes,Mt. Princeton - East Slopes,standard_ranked_requested,Mt. Princeton,Mt. Princeton,false,dry_summer,https://www.14ers.com/route.php?route=prin1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,the 2WD trailhead,reverse_ascent,false,false,published_summer_route,,,,\nco14-019,19,1.2.0,Mt. Belford - Northwest Ridge,Mt. Belford - Northwest Ridge,standard_ranked_requested,Mt. Belford,Mt. Belford,false,dry_summer,https://www.14ers.com/route.php?route=belf1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-020,20,1.2.0,Crestone Needle - South Face,Crestone Needle - South Face,standard_ranked_requested,Crestone Needle,Crestone Needle,false,dry_summer,https://www.14ers.com/route.php?route=cnee1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,SC 2WD TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-021,21,1.2.0,Mt. Yale - Southwest Slopes,Mt. Yale - Southwest Slopes,standard_ranked_requested,Mt. Yale,Mt. Yale,false,dry_summer,https://www.14ers.com/route.php?route=yale1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-022,22,1.2.0,Mt. Bross - West Slopes,Mt. Bross - West Slopes,standard_ranked_requested,Mt. Bross,Mt. Bross,false,dry_summer,https://www.14ers.com/route.php?route=bros2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-023,23,1.2.0,Kit Carson Peak - Via Challenger Point,Kit Carson Peak - Via Challenger Point,standard_ranked_requested,Kit Carson Peak,Challenger Point|Kit Carson Peak,false,dry_summer,https://www.14ers.com/route.php?route=kitc2,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,TH,published_dry_descent_return_to_start,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0.\nco14-024,24,1.2.0,Maroon Peak - South Ridge,Maroon Peak - South Ridge,standard_ranked_requested,Maroon Peak,Maroon Peak,false,dry_summer,https://www.14ers.com/route.php?route=maro1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-025,25,1.2.0,Tabeguache Peak - Via Shavano,Tabeguache Peak - Via Mt. Shavano,standard_ranked_requested,Tabeguache Peak,Mt. Shavano|Tabeguache Peak,false,dry_summer,https://www.14ers.com/route.php?route=tabe2,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,,published_dry_descent_return_to_start,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0.\nco14-026,26,1.2.0,Mt. Oxford - Via Belford,Mt. Oxford - Via Mt. Belford,standard_ranked_requested,Mt. Oxford,Mt. Belford|Mt. Oxford,false,dry_summer,https://www.14ers.com/route.php?route=oxfo2,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,Missouri Gulch,published_dry_descent_return_to_start,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0. Published total variant: via Belford standard route.\nco14-027,27,1.2.0,Mt. Sneffels - South Slopes (Yankee Boy Basin),Mt. Sneffels - South Slopes,standard_ranked_requested,Mt. Sneffels,Mt. Sneffels,false,dry_summer,https://www.14ers.com/route.php?route=snef1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD parking,reverse_ascent,false,false,published_summer_route,,,,\nco14-028,28,1.2.0,Mt. Democrat - East Slope,Mt. Democrat - East Slope,standard_ranked_requested,Mt. Democrat,Mt. Democrat,false,dry_summer,https://www.14ers.com/route.php?route=demo1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-029,29,1.2.0,Capitol Peak - Northeast Ridge (Knife Edge),Capitol Peak - Northeast Ridge,standard_ranked_requested,Capitol Peak,Capitol Peak,false,dry_summer,https://www.14ers.com/route.php?route=capi1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,the TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-030,30,1.2.0,Pikes Peak - East Slopes (Barr Trail),Pikes Peak - East Slopes,standard_ranked_requested,Pikes Peak,Pikes Peak,false,dry_summer,https://www.14ers.com/route.php?route=pikeeast,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-031,31,1.2.0,Snowmass Mountain - East Slopes,Snowmass Mountain - East Slopes,standard_ranked_requested,Snowmass Mountain,Snowmass Mountain,false,dry_summer,https://www.14ers.com/route.php?route=snow1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-032,32,1.2.0,Windom Peak - West Ridge,Windom Peak - West Ridge,standard_ranked_requested,Windom Peak,Windom Peak,false,dry_summer,https://www.14ers.com/route.php?route=wind1,,,official_identity_verified,2026-09-30,,,out_and_back,Needleton_include_Chicago_Basin_approach_exclude_train,Needleton TH,reverse_ascent,false,false,published_summer_route,,,co14-032|co14-033|co14-039|co14-058, Shared Needleton/Chicago Basin approach: keep this connected group in one evaluation split. Individual summit itinerary retained; custom combination excluded.\nco14-033,33,1.2.0,Mt. Eolus - Northeast Ridge,Mt. Eolus - Northeast Ridge,standard_ranked_requested,Mt. Eolus,Mt. Eolus,false,dry_summer,https://www.14ers.com/route.php?route=eolu3,,,official_identity_verified,2026-09-30,,,out_and_back,Needleton_include_Chicago_Basin_approach_exclude_train,Needleton TH,reverse_ascent,false,false,published_summer_route,,,co14-032|co14-033|co14-039|co14-058, Shared Needleton/Chicago Basin approach: keep this connected group in one evaluation split. Individual summit itinerary retained; custom combination excluded.\nco14-034,34,1.2.0,Challenger Point - North Slope,Challenger Point - North Slope,standard_ranked_requested,Challenger Point,Challenger Point,false,dry_summer,https://www.14ers.com/route.php?route=chal1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-035,35,1.2.0,Mt. Columbia - West Slopes,Mt. Columbia - West Slopes,standard_ranked_requested,Mt. Columbia,Mt. Columbia,false,dry_summer,https://www.14ers.com/route.php?route=colu2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-036,36,1.2.0,Missouri Mountain - Northwest Ridge,Missouri Mountain - Northwest Ridge,standard_ranked_requested,Missouri Mountain,Missouri Mountain,false,dry_summer,https://www.14ers.com/route.php?route=miss1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-037,37,1.2.0,Humboldt Peak - West Ridge,Humboldt Peak - West Ridge,standard_ranked_requested,Humboldt Peak,Humboldt Peak,false,dry_summer,https://www.14ers.com/route.php?route=humb1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,SC 2WD TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-038,38,1.2.0,Mt. Bierstadt - West Slopes,Mt. Bierstadt - West Slopes,standard_ranked_requested,Mt. Bierstadt,Mt. Bierstadt,false,dry_summer,https://www.14ers.com/route.php?route=bier1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-039,39,1.2.0,Sunlight Peak - South Face,Sunlight Peak - South Face,standard_ranked_requested,Sunlight Peak,Sunlight Peak,false,dry_summer,https://www.14ers.com/route.php?route=sunl2,,,official_identity_verified,2026-09-30,,,out_and_back,Needleton_include_Chicago_Basin_approach_exclude_train,Needleton TH,reverse_ascent,false,false,published_summer_route,,,co14-032|co14-033|co14-039|co14-058, Shared Needleton/Chicago Basin approach: keep this connected group in one evaluation split. Individual summit itinerary retained; custom combination excluded.\nco14-040,40,1.2.0,Handies Peak - Southwest Slopes,Handies Peak - Southwest Slopes,standard_ranked_requested,Handies Peak,Handies Peak,false,dry_summer,https://www.14ers.com/route.php?route=hand1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,the bottom of the American Basin road,reverse_ascent,false,false,published_summer_route,,,,\nco14-041,41,1.2.0,Culebra Peak - Northwest Ridge,Culebra Peak - Northwest Ridge,standard_ranked_requested,Culebra Peak,Culebra Peak,false,dry_summer,https://www.14ers.com/route.php?route=cule1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,ranch HQ building,reverse_ascent,false,false,published_summer_route,,,,\nco14-042,42,1.2.0,Mt. Lindsey - Northwest Gully,Mt. Lindsey - Northwest Gully,standard_ranked_requested,Mt. Lindsey,Mt. Lindsey,false,dry_summer,https://www.14ers.com/route.php?route=lind1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-043,43,1.2.0,Ellingwood Point - South Face,Ellingwood Point - South Face,standard_ranked_requested,Ellingwood Point,Ellingwood Point,false,dry_summer,https://www.14ers.com/route.php?route=elli2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,"8,000",reverse_ascent,false,false,published_summer_route,,,,\nco14-044,44,1.2.0,Little Bear Peak - West Ridge and Hourglass,Little Bear Peak - West Ridge and Hourglass,standard_ranked_requested,Little Bear Peak,Little Bear Peak,false,dry_summer,https://www.14ers.com/route.php?route=litt6,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,"8,000",reverse_ascent,false,false,published_summer_route,,,,\nco14-045,45,1.2.0,Mt. Sherman - Southwest Ridge,Mt. Sherman - Southwest Ridge,standard_ranked_requested,Mt. Sherman,Mt. Sherman,false,dry_summer,https://www.14ers.com/route.php?route=sher1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,"the 12,000-foot gate",reverse_ascent,false,false,published_summer_route,,,,\nco14-046,46,1.2.0,Redcloud Peak - Northeast Ridge,Redcloud Peak - Northeast Ridge,standard_ranked_requested,Redcloud Peak,Redcloud Peak,false,dry_summer,https://www.14ers.com/route.php?route=redc1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-047,47,1.2.0,Pyramid Peak - Northeast Ridge,Pyramid Peak - Northeast Ridge,standard_ranked_requested,Pyramid Peak,Pyramid Peak,false,dry_summer,https://www.14ers.com/route.php?route=pyra1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-048,48,1.2.0,Wilson Peak - Southwest Ridge,Wilson Peak - Southwest Ridge,standard_ranked_requested,Wilson Peak,Wilson Peak,false,dry_summer,https://www.14ers.com/route.php?route=wils1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Navajo Basin TH,reverse_ascent,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0.\nco14-049,49,1.2.0,Wetterhorn Peak - Southeast Ridge,Wetterhorn Peak - Southeast Ridge,standard_ranked_requested,Wetterhorn Peak,Wetterhorn Peak,false,dry_summer,https://www.14ers.com/route.php?route=wett1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-050,50,1.2.0,San Luis Peak - Northeast Ridge,San Luis Peak - Northeast Ridge,standard_ranked_requested,San Luis Peak,San Luis Peak,false,dry_summer,https://www.14ers.com/route.php?route=sanl1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-051,51,1.2.0,Mt. of the Holy Cross - North Ridge,Mt. of the Holy Cross - North Ridge,standard_ranked_requested,Mt. of the Holy Cross,Mt. of the Holy Cross,false,dry_summer,https://www.14ers.com/route.php?route=holy1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-052,52,1.2.0,Huron Peak - Northwest Slopes,Huron Peak - Northwest Slopes,standard_ranked_requested,Huron Peak,Huron Peak,false,dry_summer,https://www.14ers.com/route.php?route=huro1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-053,53,1.2.0,Sunshine Peak - Via Redcloud,Sunshine Peak - Via Redcloud Peak,standard_ranked_requested,Sunshine Peak,Redcloud Peak|Sunshine Peak,false,dry_summer,https://www.14ers.com/route.php?route=suns2,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,,published_dry_descent_return_to_start,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0.\nco14-054,54,1.2.0,Mt. Cameron - West Ridge,Mt. Cameron - West Ridge,standard_named_requested,Mt. Cameron,Mt. Cameron,false,dry_summer,https://www.14ers.com/route.php?route=came1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-055,55,1.2.0,North Maroon Peak - Northeast Ridge,North Maroon Peak - Northeast Ridge,standard_named_requested,North Maroon Peak,North Maroon Peak,false,dry_summer,https://www.14ers.com/route.php?route=nmar2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-056,56,1.2.0,El Diente Peak - South Slopes,El Diente Peak - South Slopes,standard_named_requested,El Diente Peak,El Diente Peak,false,dry_summer,https://www.14ers.com/route.php?route=eldi4,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-057,57,1.2.0,Conundrum Peak - South Ridge,Conundrum Peak - South Ridge,standard_named_requested,Conundrum Peak,Conundrum Peak,false,dry_summer,https://www.14ers.com/route.php?route=conu1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,the main trailhead,reverse_ascent,false,false,published_summer_route,,,co14-057|co14-088,Related itineraries retained per user; keep the whole connected group in one evaluation split.\nco14-058,58,1.2.0,North Eolus - South Ridge,North Eolus - South Ridge,standard_named_requested,North Eolus,North Eolus,false,dry_summer,https://www.14ers.com/route.php?route=neol4,,,official_identity_verified,2026-09-30,,,out_and_back,Needleton_include_Chicago_Basin_approach_exclude_train,Needleton TH,reverse_ascent,false,false,published_summer_route,,,co14-032|co14-033|co14-039|co14-058, Shared Needleton/Chicago Basin approach: keep this connected group in one evaluation split. Individual summit itinerary retained; custom combination excluded.\nco14-059,59,1.2.0,Mt. Elbert - East Ridge,Mt. Elbert - East Ridge,alternate_requested,,Mt. Elbert,false,dry_summer,https://www.14ers.com/route.php?route=elbe2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD paved TH,reverse_ascent,false,false,published_summer_route,,,,\nco14-060,60,1.2.0,Mt. Elbert - Black Cloud Trail,Mt. Elbert - Southeast Ridge,alternate_requested,,Mt. Elbert,false,dry_summer,https://www.14ers.com/route.php?route=elbe3,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,Black Cloud Trail maps to the official Southeast Ridge; original requested name preserved.\nco14-061,61,1.2.0,Quandary Peak - West Ridge,Quandary Peak - West Ridge,alternate_requested,,Quandary Peak,false,dry_summer,https://www.14ers.com/route.php?route=quan3,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Blue Lakes,West Ridge to Blue Lakes; no Cristo Couloir,false,false,published_summer_route,,,,Dry baseline descends the West Ridge ascent route; exclude Cristo Couloir snow descent. Published total variant: if you descend the ascent route.\nco14-062,62,1.2.0,Torreys Peak - Kelso Ridge,Torreys Peak - Kelso Ridge,alternate_requested,,Torreys Peak,false,dry_summer,https://www.14ers.com/route.php?route=torr4,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-063,63,1.2.0,Mt. Bierstadt to Blue Sky - The Sawtooth Traverse,Bierstadt+Sawtooth+Blue Sky,alternate_requested,,Mt. Bierstadt|Mt. Blue Sky,false,dry_summer,https://www.14ers.com/route.php?route=bier4,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,,published_dry_descent_return_to_start,false,false,published_summer_route,,,,"Full Bierstadt/Sawtooth/Blue Sky itinerary from Guanella Pass, not just the summit-to-summit connector."\nco14-064,64,1.2.0,Mt. Massive - Southwest Slopes,Mt. Massive - Southwest Slopes,alternate_requested,,Mt. Massive,false,dry_summer,https://www.14ers.com/route.php?route=mass2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-065,65,1.2.0,Grays & Torreys - Combination Route,Grays and Torreys,alternate_requested,,Grays Peak|Torreys Peak,false,dry_summer,https://www.14ers.com/route.php?route=torr5,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,,published_dry_descent_return_to_start,false,false,published_summer_route,,,,\nco14-066,66,1.2.0,"The Decalibron Loop (Democrat, Cameron, Lincoln, Bross)",Decalibron,alternate_requested,,Mt. Democrat|Mt. Cameron|Mt. Lincoln|Mt. Bross,false,dry_summer,https://www.14ers.com/route.php?route=bros6,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,,published_dry_descent_return_to_start,false,false,published_summer_route,,,,"Keep all four summit targets; Bross summit access must be recorded, not inferred from the route name."\nco14-067,67,1.2.0,Longs Peak - The Loft,Longs Peak - Loft Route,alternate_requested,,Longs Peak,false,dry_summer,https://www.14ers.com/route.php?route=long2,,,official_identity_verified,2026-09-30,,,loop,lowest_published_summer_start,Longs Peak Trailhead,Keyhole Route to Longs Peak Trailhead,false,false,published_summer_route,,,,Former exact-source alias replaced in manifest 1.2.0.\nco14-068,68,1.2.0,Longs Peak - Clark\'s Arrow,Mt. Bierstadt - East Ridge,alternate_requested,,Mt. Bierstadt,true,dry_summer,https://www.14ers.com/route.php?route=bier3,https://www.14ers.com/php14ers/tripreport.php?trip=23510,,official_identity_verified,2026-09-30,,,out_and_back,published_overview_start,Blue Sky Summit,reverse_ascent,false,false,published_summer_route,,,,"Replaces Longs Peak - Loft Route under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-069,69,1.2.0,Longs Peak - Kiener\'s Route,Crestone Needle - Ellingwood Arete/Ledges,alternate_requested,,Crestone Needle,true,dry_summer,https://www.14ers.com/route.php?route=cnee5,https://www.14ers.com/php14ers/tripreport.php?trip=15230,https://www.14ers.com/route.php?route=cnee1,official_identity_verified,2026-09-30,,,loop,published_overview_start,South Colony Lakes,standard_South_Face_via_Broken_Hand_Pass,false,false,published_summer_route,,,,"Replaces Longs Peak - Kiener\'s Route under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage. Ledges start selected to avoid the direct-start snowfield; GPX branch, dry crossing and South Face descent still require review. Class 5 is observed in official source; encoding 16 retained without override."\nco14-070,70,1.2.0,Mt. Sneffels - Southwest Ridge,Mt. Sneffels - Southwest Ridge,alternate_requested,,Mt. Sneffels,false,dry_summer,https://www.14ers.com/route.php?route=snef2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD parking,reverse_ascent,false,false,published_summer_route,,,,\nco14-071,71,1.2.0,Mt. Yale - East Ridge,Mt. Yale - East Ridge,alternate_requested,,Mt. Yale,false,dry_summer,https://www.14ers.com/route.php?route=yale2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-072,72,1.2.0,La Plata Peak - Southwest Ridge,La Plata Peak - Southwest Ridge,alternate_requested,,La Plata Peak,false,dry_summer,https://www.14ers.com/route.php?route=lapl2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-073,73,1.2.0,Pikes Peak - Northwest Slopes (The Crags),Pikes Peak - Northwest Slopes,alternate_requested,,Pikes Peak,false,dry_summer,https://www.14ers.com/route.php?route=pike1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-074,74,1.2.0,Mt. of the Holy Cross - Halo Ridge,Mt. of the Holy Cross - Halo Ridge,alternate_requested,,Mt. of the Holy Cross,false,dry_summer,https://www.14ers.com/route.php?route=holy3,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-075,75,1.2.0,Handies Peak - East Slopes (Grizzly Gulch),Handies Peak - East Slopes,alternate_requested,,Handies Peak,false,dry_summer,https://www.14ers.com/route.php?route=hand2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-076,76,1.2.0,Mt. Shavano & Tabeguache - Combination,Tabeguache Peak - West Ridge,alternate_requested,,Tabeguache Peak,true,dry_summer,https://www.14ers.com/route.php?route=tabe3w,https://www.14ers.com/php14ers/tripreport.php?trip=10380,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Jennings Creek,reverse_ascent,false,false,published_summer_route,,,,"Replaces Tabeguache Peak - Via Mt. Shavano under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-077,77,1.2.0,Mt. Belford & Oxford - Combination,La Plata Peak - Ellingwood Ridge,alternate_requested,,La Plata Peak,true,dry_summer,https://www.14ers.com/route.php?route=lapl3,https://www.14ers.com/php14ers/tripreport.php?trip=23669,https://www.14ers.com/route.php?route=lapl1,official_identity_verified,2026-09-30,,,loop,lowest_published_summer_start,La Plata Peak,Northwest_Ridge_return_to_La_Plata_Peak_trailhead,false,false,published_summer_route,,,,"Replaces Mt. Oxford - Via Mt. Belford under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-078,78,1.2.0,Redcloud & Sunshine - Combination,Grays Peak - South Ridge,alternate_requested,,Grays Peak,true,dry_summer,https://www.14ers.com/route.php?route=gray3,https://www.14ers.com/php14ers/tripreport.php?trip=20351,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Argentine Pass,reverse_ascent,false,false,published_summer_route,,,,"Replaces Sunshine Peak - Via Redcloud Peak under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-079,79,1.2.0,Crestone Traverse (Peak to Needle),Crestones Traverse,alternate_requested,,Crestone Peak|Crestone Needle,false,dry_summer,https://www.14ers.com/route.php?route=cnee3,,,official_identity_verified,2026-09-30,,,traverse_return,lowest_published_summer_start,SC 2WD TH,second_peak_standard_dry_descent_return_to_start,false,false,published_summer_route,,,,Peak-to-Needle direction. Published connector is not automatically a complete trailhead-to-trailhead GPX.\nco14-080,80,1.2.0,Little Bear to Blanca Traverse,Little Bear + Blanca Traverse,alternate_requested,,Little Bear Peak|Blanca Peak,false,dry_summer,https://www.14ers.com/route.php?route=litt3,,,official_identity_verified,2026-09-30,,,traverse_return,lowest_published_summer_start,"8,000",second_peak_standard_dry_descent_return_to_start,false,false,published_summer_route,,,,Little Bear-to-Blanca direction; include approach and standard Blanca descent.\nco14-081,81,1.2.0,El Diente to Mt. Wilson Traverse,Mt. Wilson + El Diente Traverse,alternate_requested,,El Diente Peak|Mt. Wilson,false,dry_summer,https://www.14ers.com/route.php?route=mwil5,,,official_identity_verified,2026-09-30,,,traverse_return,lowest_published_summer_start,,second_peak_standard_dry_descent_return_to_start,false,false,published_summer_route,,,,El Diente-to-Wilson direction; published connector needs approach and descent coverage checks.\nco14-082,82,1.2.0,Maroon Bells Traverse (Maroon to North Maroon),Bells Traverse,alternate_requested,,Maroon Peak|North Maroon Peak,false,dry_summer,https://www.14ers.com/route.php?route=nmar4,,,official_identity_verified,2026-09-30,,,traverse_return,lowest_published_summer_start,,second_peak_standard_dry_descent_return_to_start,false,false,published_summer_route,,,,Maroon-to-North Maroon direction; preserve raw Class 5 if source assigns it.\nco14-083,83,1.2.0,Blanca & Ellingwood - Combination,Blanca and Ellingwood,alternate_requested,,Blanca Peak|Ellingwood Point,false,dry_summer,https://www.14ers.com/route.php?route=elli3,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,"8,000",published_dry_descent_return_to_start,false,false,published_summer_route,,,,\nco14-084,84,1.2.0,Wilson Peak - Rock of Ages Saddle,Mt. Blue Sky - West Ridge from Guanella Pass,alternate_requested,,Mt. Blue Sky,true,dry_summer,https://www.14ers.com/route.php?route=evan3,https://www.14ers.com/php14ers/tripreport.php?trip=5221,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Guanella Pass,reverse_ascent,false,false,published_summer_route,,,,"Replaces Wilson Peak - Southwest Ridge under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-085,85,1.2.0,Mt. Sherman - East Ridge,Mt. Sherman - West Slopes,alternate_requested,,Mt. Sherman,true,dry_summer,https://www.14ers.com/route.php?route=sher2,,,official_identity_verified,2026-09-30,,,out_and_back,published_summer_trailhead,Iowa Gulch,reverse_ascent,false,false,published_summer_route,,,,"User-authorized replacement of Sherman East Ridge: West Slopes via Iowa Gulch, official dry-season route."\nco14-086,86,1.2.0,Huron Peak - Lulu Gulch,Huron Peak - North Ridge from Lulu Gulch,alternate_requested,,Huron Peak,false,dry_summer,https://www.14ers.com/route.php?route=huro4,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-087,87,1.2.0,Missouri Mountain - North Face,Missouri Mountain - West Ridge,alternate_requested,,Missouri Mountain,true,dry_summer,https://www.14ers.com/route.php?route=miss2,,,official_identity_verified,2026-09-30,,,out_and_back,lower_Rockdale_2WD_include_road_walk,Rockdale 2WD,reverse_ascent,false,false,published_summer_route,,,,"User-authorized replacement of Missouri North Face: West Ridge via Clohesy Lake. Baseline starts at lower Rockdale 2WD trailhead, not the upper 4WD gate."\nco14-088,88,1.2.0,Castle & Conundrum - Combination,Castle and Conundrum,alternate_requested,,Castle Peak|Conundrum Peak,false,dry_summer,https://www.14ers.com/route.php?route=cast4,,,official_identity_verified,2026-09-30,,,multi_peak_return,lowest_published_summer_start,the main trailhead,published_dry_descent_return_to_start,false,false,published_summer_route,,,co14-057|co14-088,Related itineraries retained per user; keep the whole connected group in one evaluation split.\nco14-089,89,1.2.0,"Chicago Basin 3-Combo (Eolus, Windom, Sunlight)",Grays Peak - Southwest Ridge,alternate_requested,,Grays Peak,true,dry_summer,https://www.14ers.com/route.php?route=gray8,https://www.14ers.com/php14ers/tripreport.php?trip=14394,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Chihuahua Gulch,reverse_ascent,false,false,published_summer_route,,,,"Replaces Chicago Basin 3-Combo (Eolus, Windom, Sunlight) under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-090,90,1.2.0,Mt. Princeton - Southwest Ridge,Torreys Peak - West Ridge,alternate_requested,,Grizzly Peak D|Torreys Peak|Grizzly Peak D,true,dry_summer,https://www.14ers.com/route.php?route=torr11,https://www.14ers.com/php14ers/tripreport.php?trip=3289,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Loveland Pass,reverse_ascent,false,false,published_summer_route,,,,"Replaces Mt. Princeton - Southwest Ridge under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-091,91,1.2.0,Mt. Columbia - Southeast Ridge,Mt. Columbia - Southeast Ridge,alternate_requested,,Mt. Columbia,false,dry_summer,https://www.14ers.com/route.php?route=colu5,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-092,92,1.2.0,Mt. Harvard & Columbia - Traverse,Harvard and Columbia Traverse,alternate_requested,,Mt. Harvard|Mt. Columbia,false,dry_summer,https://www.14ers.com/route.php?route=harv4,,,official_identity_verified,2026-09-30,,,traverse_return,lowest_published_summer_start,TH,second_peak_standard_dry_descent_return_to_start,false,false,published_summer_route,,,,\nco14-093,93,1.2.0,Challenger & Kit Carson - Combination,Kit Carson Peak - East Ridge,alternate_requested,,Kit Carson Peak,true,dry_summer,https://www.14ers.com/route.php?route=kitc3,https://www.14ers.com/php14ers/tripreport.php?trip=9208,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,2WD SC TH,reverse_ascent,false,false,published_summer_route,,,,"Replaces Kit Carson Peak - Via Challenger Point under user instruction to exclude snow-dependent routes/custom Chicago Basin combo and reach 100 distinct routes. Trip report supports route activity, not a measured popularity rank or exact GPX coverage."\nco14-094,94,1.2.0,Wetterhorn Peak - East Ridge,Mt. Belford - Southwest Slopes,alternate_requested,,Mt. Belford,true,dry_summer,https://www.14ers.com/route.php?route=belf9,,,official_identity_verified,2026-09-30,,,out_and_back,published_summer_trailhead,Missouri Gulch,reverse_ascent,false,false,published_summer_route,,,,"User authorized replacing #94 after exact Wetterhorn East Ridge GPX was not located; selected official summer Southwest Slopes via Elkhead Pass, reverse ascent, source belf9. Original requested name retained."\nco14-095,95,1.2.0,Uncompahgre Peak - East Ridge,Kit Carson Peak - North Ridge,alternate_requested,,Kit Carson Peak|Challenger Point,true,dry_summer,https://www.14ers.com/route.php?route=kitc4,,,official_identity_verified,2026-09-30,,,loop,lowest_published_summer_start,Willow Creek,Via Challenger Point to Willow Creek,false,false,published_summer_route,,,,"User-authorized replacement of Uncompahgre East Ridge: Kit Carson North Ridge, official Class 4 route; descent via Challenger to Willow Creek."\nco14-096,96,1.2.0,Snowmass Mountain - West Slopes,Snowmass Mountain - West Slope,alternate_requested,,Snowmass Mountain,false,dry_summer,https://www.14ers.com/route.php?route=snow2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-097,97,1.2.0,Mt. Massive - Tour de Abyss,Mt. Bierstadt & Mt. Blue Sky - Tour de Abyss Loop,alternate_requested,,Mt. Bierstadt|Mt. Blue Sky,false,dry_summer,https://www.14ers.com/route.php?route=bier5,,,official_identity_verified,2026-09-30,,,loop,Blue_Sky_Summit_parking_include_return_walk,Blue Sky Summit parking,Return on foot to Blue Sky Summit parking,false,false,published_summer_route,,,,"Retain Tour de Abyss; correct peak identity to Bierstadt and Blue Sky. Baseline full loop from Blue Sky Summit parking, including return walking segment."\nco14-098,98,1.2.0,Mt. Blue Sky - East Ridge (from Echo Lake),Mt. Blue Sky - West Ridge from Echo Lake,alternate_requested,,Mt. Blue Sky,false,dry_summer,https://www.14ers.com/route.php?route=evan1,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,Echo Lake,reverse_ascent,false,false,published_summer_route,,,,Echo Lake itinerary uses published West Ridge from Echo Lake. Correct directional label; preserve original request.\nco14-099,99,1.2.0,San Luis Peak - South Ridge,San Luis Peak - South Ridge,alternate_requested,,San Luis Peak,false,dry_summer,https://www.14ers.com/route.php?route=sanl2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\nco14-100,100,1.2.0,Mt. Antero - Little Browns Creek,Mt. Antero - Via Little Browns Creek,alternate_requested,,Mt. Antero,false,dry_summer,https://www.14ers.com/route.php?route=ante2,,,official_identity_verified,2026-09-30,,,out_and_back,lowest_published_summer_start,,reverse_ascent,false,false,published_summer_route,,,,\n'
MANIFEST_PATH = Path(DATA_ROOT) / 'route_manifest.csv'
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
if not MANIFEST_PATH.exists():
    MANIFEST_PATH.write_text(MANIFEST_CSV, encoding='utf-8')
manifest_preview = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
assert len(manifest_preview) == 100
manifest_preview[['route_id', 'requested_name', 'review_note']]

## Execution configuration

For live access, set the confirmations only after documented source permission and accepted download agreement; use your own private authorized cookies if needed. Begin with `ONLY_ROUTES = ['co14-001', 'co14-095', 'co14-097']`, inspect the raw HTML/GPX, then collect the full roster. No automated login or verification bypass is implemented.

For an owner-approved local export, set `CURATED_FILE` to its private JSON path and `OFFLINE=True`. The export must carry permission evidence, the matching manifest hash, reviewed metadata/scope and GPX checksums; see README.md for the contract. Offline mode makes no network requests and does not require live-download flags.

To import already accepted normal browser downloads, set `BROWSER_DOWNLOAD_DIR` to the private receipt directory and `OFFLINE=True`, leaving `CURATED_FILE=None`. Hashes and source identity are validated, but scope verification flags remain unchanged. Original ascent tracks are not automatically complete itineraries.

The default flags remain false. Reports preserve all 100 IDs, including pending and smoke-subset rows. Software fixture tests do not establish a complete dataset.

In [ ]:
CONTACT = 'replace-with-your-research-contact-or-project-url'
SOURCE_PERMISSION_CONFIRMED = False
DOWNLOAD_AGREEMENT_ACCEPTED = False
COOKIE_FILE = None
REFRESH = False
CURATED_FILE = None  # Private approved-export JSON; see README.md.
BROWSER_DOWNLOAD_DIR = None  # Private normal-browser GPX/metadata and hashed receipts; OFFLINE=True.
OFFLINE = False  # True imports local exports without making network requests.
ONLY_ROUTES = None  # Smoke test: ['co14-001', 'co14-095', 'co14-097']

if SOURCE_PERMISSION_CONFIRMED and CONTACT.startswith('replace-'):
    raise ValueError('Set CONTACT to your real research contact or project URL.')

summary = run(
    manifest=MANIFEST_PATH,
    out=Path(DATA_ROOT) / 'raw',
    contact=CONTACT,
    source_permission_confirmed=SOURCE_PERMISSION_CONFIRMED,
    download_agreement_accepted=DOWNLOAD_AGREEMENT_ACCEPTED,
    cookie_file=COOKIE_FILE,
    refresh=REFRESH,
    curated_file=CURATED_FILE,
    offline=OFFLINE,
    only_routes=ONLY_ROUTES,
    browser_download_dir=BROWSER_DOWNLOAD_DIR,
)
print(json.dumps(summary, indent=2))
status = pd.read_csv(Path(DATA_ROOT) / 'raw' / 'route_status.csv', keep_default_na=False)
display(status[['route_id', 'canonical_name', 'status', 'metadata_scope_status', 'error']]
        if 'metadata_scope_status' in status else status[['route_id', 'canonical_name', 'status', 'error']])


## Review and rerun

`missing_routes.csv` tracks unavailable downloads; `review_required.csv` additionally includes duplicate and unverified records. Preserve `REFRESH=False` to reuse cached successful requests. When the site blocks access, resolve the access issue through the owner or use an authorized export; repeated retries or alternative guessed endpoints are not a fix.

Example manifest edit (after verifying the route against the actual source):
```python
edited = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
# Example only: advertised Elbert Northeast Ridge source URL.
edited.loc[edited.route_id == 'co14-001', 'source_url'] = \
    'https://www.14ers.com/route.php?route=elbe1'
edited.to_csv(MANIFEST_PATH, index=False)
```
Rerun the execution cell after your edits. #97 is retained as the Bierstadt–Blue Sky Tour de Abyss. #87 is Missouri West Ridge, #85 is Sherman West Slopes, and #95 is Kit Carson North Ridge. Class 5 encodes as 16. Keep stable IDs and document future changes in project_config.json and the roadmap.

In [ ]:
# Capture successful-run dependency versions for the repository handoff.
import importlib.metadata as metadata
versions = {name: metadata.version(name) for name in
            ['requests', 'beautifulsoup4', 'pandas', 'defusedxml']}
write_json(Path(DATA_ROOT) / 'dependency_versions.json', versions)
print(versions)
print('Reports and private downloads:', Path(DATA_ROOT) / 'raw')

## Phase 3 — recorded-track features and QA

Use percent grade: medium 20–35%, extreme >35%. Block 2 computes steep mileage, longest continuous steep runs, all five source encodings, sensitivity and approximate overlap groups. Complete-itinerary geometry fields remain empty until scope and QA pass. Version 3.1 adds accepted-segment density observations for all 100 tracks with separate recorded-chord bounds; strict densities below 95% valid distance remain missing. Phase Four local exploration and Phase Five preparation are documented in the README/roadmap.

Extract the current kickoff ZIP to a project directory and set `FEATURE_PROJECT_ROOT` below. The package includes the project config, Block 1/2 and snapshot auditor. Keep the already acquired private snapshot at `DATA_ROOT/raw`; the ZIP contains no private GPX. This cell does not assemble or certify missing approaches/returns. Feature dictionaries and parameter choices live in the package's project config.


In [ ]:
FEATURE_SOURCE = '"""Colorado 14er Phase 3: segment-safe recorded-track features and QA.\n\nPython 3.10+, numpy and the existing block1_scraper runtime dependencies.\nNo HTTP requests, itinerary assembly, timestamp features, or automatic scope approval.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nfrom defusedxml import ElementTree as ET\n\nFEATURE_VERSION = \'3.1.0\'\nM_PER_MILE = 1609.344\nFT_PER_M = 3.280839895\nGEOMETRY_FEATURES = (\'distance_mi\', \'gain_ft\', \'crux_density_medium\',\n                     \'crux_density_extreme\', \'crux_medium_mi\', \'crux_extreme_mi\',\n                     \'steep_above20_mi\', \'longest_medium_mi\', \'longest_extreme_mi\',\n                     \'longest_above20_mi\')\nRISKS = (\'exposure\', \'rockfall\', \'route_finding\', \'commitment\')\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef distance(a, b):\n    p, q = math.radians(a[0]), math.radians(b[0])\n    h = math.sin((q-p)/2)**2 + math.cos(p)*math.cos(q)*math.sin(math.radians(b[1]-a[1])/2)**2\n    return 6371008.8 * 2 * math.asin(math.sqrt(min(1.0, max(0.0, h))))\n\n\ndef sequences(body):\n    """Read only track/route points, keeping every GPX sequence boundary."""\n    root = ET.fromstring(body)\n    tag = lambda n: n.tag.rsplit(\'}\', 1)[-1]\n    result = []\n    for node in root.iter():\n        if tag(node) not in (\'trkseg\', \'rte\'):\n            continue\n        points = []\n        for point in node:\n            if tag(point) not in (\'trkpt\', \'rtept\'):\n                continue\n            elevation = next((float(c.text) for c in point if tag(c) == \'ele\'), None)\n            points.append((float(point.attrib[\'lat\']), float(point.attrib[\'lon\']), elevation))\n        if len(points) >= 2:\n            result.append(points)\n    return result\n\n\ndef validated_runs(seqs, params):\n    """Invalid edges split profiles; no interpolation or crux run crosses them."""\n    runs, counts, total, accepted = [], Counter(), 0.0, 0.0\n    for seq in seqs:\n        run = [seq[0]]\n        for a, b in zip(seq, seq[1:]):\n            d = distance(a, b)\n            total += d\n            reason = None\n            if d < params[\'min_horizontal_step_m\']:\n                reason = \'near_zero_step\'\n            elif d > params[\'max_horizontal_step_m\'] and not math.isclose(\n                    d, params[\'max_horizontal_step_m\'], rel_tol=0, abs_tol=1e-7):\n                reason = \'long_gap\'\n            elif a[2] is None or b[2] is None:\n                reason = \'missing_elevation\'\n            elif abs(b[2]-a[2])/d * 100 > params[\'max_abs_grade_pct\']:\n                reason = \'implausible_grade\'\n            if reason:\n                counts[reason] += 1\n                if len(run) >= 2:\n                    runs.append(run)\n                run = [b]\n            else:\n                accepted += d\n                run.append(b)\n        if len(run) >= 2:\n            runs.append(run)\n    return runs, {\'recorded_distance_m\': total, \'accepted_distance_m\': accepted,\n                  \'valid_distance_fraction\': accepted/total if total else 0.0,\n                  **{k: counts[k] for k in (\'near_zero_step\', \'long_gap\', \'missing_elevation\', \'implausible_grade\')}}\n\n\ndef accepted_segment_densities(profile, qa):\n    """Describe observed segments; bound unknown excluded chord lengths, never fill them."""\n    fraction = qa[\'valid_distance_fraction\']\n    result = {\'observed_segment_scope\': \'accepted_recorded_segments_only\',\n              \'excluded_recorded_distance_fraction\': 1-fraction}\n    for band in (\'medium\', \'extreme\'):\n        value = profile[\'crux_density_\'+band]\n        prefix = \'observed_segment_crux_density_\'+band\n        result[prefix] = value\n        # Chord-distance bounds conditional on the recorded polyline, not true terrain length.\n        result[\'recorded_chord_crux_density_\'+band+\'_lower_pct\'] = value*fraction if value is not None else None\n        result[\'recorded_chord_crux_density_\'+band+\'_upper_pct\'] = min(100, value*fraction+100*(1-fraction)) if value is not None else None\n    return result\n\n\ndef smooth_profile(x, z, window):\n    """Distance-window mean with linear endpoint extension; keeps a linear slope."""\n    if window <= 0 or len(x) < 3:\n        return z.copy()\n    step = x[1]-x[0]\n    half = int(math.floor(window/(2*step)))\n    if half == 0:\n        return z.copy()\n    left = z[0] - np.arange(half, 0, -1) * (z[1]-z[0])\n    right = z[-1] + np.arange(1, half+1) * (z[-1]-z[-2])\n    return np.convolve(np.concatenate((left, z, right)), np.ones(2*half+1)/(2*half+1), mode=\'valid\')\n\n\ndef profile_edges(runs, interval, window):\n    edges = []\n    for run in runs:\n        dx = np.array([distance(a, b) for a, b in zip(run, run[1:])])\n        x = np.concatenate(([0.0], np.cumsum(dx)))\n        z = np.array([p[2] for p in run], dtype=float)\n        if interval:\n            # Equal spacing includes both endpoints and never exceeds the chosen interval.\n            grid = np.linspace(0.0, x[-1], max(1, math.ceil(x[-1]/interval))+1)\n            z = np.interp(grid, x, z)\n            x = grid\n        z = smooth_profile(x, z, window)\n        edges.append((np.diff(x), np.diff(z)))\n    return edges\n\n\ndef longest_run(dx, mask):\n    longest = current = 0.0\n    for d, active in zip(dx, mask):\n        current = current + float(d) if active else 0.0\n        longest = max(longest, current)\n    return longest\n\n\ndef summarize_edges(edges, params):\n    totals = Counter()\n    longest = Counter()\n    for dx, dz in edges:\n        grade = np.abs(dz/dx)*100\n        # Avoid numerical jitter assigning an exact boundary to different bands.\n        for boundary in (params[\'medium_grade_min_pct\'], params[\'extreme_grade_min_pct\']):\n            grade[np.isclose(grade, boundary, rtol=0, atol=1e-9)] = boundary\n        medium = (grade >= params[\'medium_grade_min_pct\']) & (grade <= params[\'extreme_grade_min_pct\'])\n        extreme = grade > params[\'extreme_grade_min_pct\']\n        above20 = medium | extreme\n        totals[\'distance\'] += float(dx.sum())\n        totals[\'gain\'] += float(np.maximum(dz, 0).sum())\n        for key, mask in [(\'medium\', medium), (\'extreme\', extreme), (\'above20\', above20)]:\n            totals[key] += float(dx[mask].sum())\n            longest[key] = max(longest[key], longest_run(dx, mask))\n    length = totals[\'distance\']\n    return {\'distance_mi\': length/M_PER_MILE if length else None,\n            \'gain_ft\': totals[\'gain\']*FT_PER_M if length else None,\n            \'crux_density_medium\': 100*totals[\'medium\']/length if length else None,\n            \'crux_density_extreme\': 100*totals[\'extreme\']/length if length else None,\n            \'crux_medium_mi\': totals[\'medium\']/M_PER_MILE if length else None,\n            \'crux_extreme_mi\': totals[\'extreme\']/M_PER_MILE if length else None,\n            \'steep_above20_mi\': totals[\'above20\']/M_PER_MILE if length else None,\n            \'longest_medium_mi\': longest[\'medium\']/M_PER_MILE if length else None,\n            \'longest_extreme_mi\': longest[\'extreme\']/M_PER_MILE if length else None,\n            \'longest_above20_mi\': longest[\'above20\']/M_PER_MILE if length else None}\n\n\ndef analyze_track(seqs, params):\n    runs, qa = validated_runs(seqs, params)\n    raw = summarize_edges(profile_edges(runs, None, 0), params)\n    profile = summarize_edges(profile_edges(runs, params[\'resample_interval_m\'], params[\'smoothing_window_m\']), params)\n    sensitivity = []\n    for interval in params[\'sensitivity_intervals_m\']:\n        for window in params[\'sensitivity_windows_m\']:\n            sensitivity.append(summarize_edges(profile_edges(runs, interval, window), params))\n    for key, field in [(\'crux_density_medium\', \'density_medium_range_pp\'),\n                       (\'crux_density_extreme\', \'density_extreme_range_pp\'), (\'gain_ft\', \'gain_range_ft\')]:\n        values = [x[key] for x in sensitivity if x[key] is not None]\n        qa[field] = max(values)-min(values) if values else None\n    qa[\'gain_sensitivity_pct\'] = 100*qa[\'gain_range_ft\']/profile[\'gain_ft\'] if profile[\'gain_ft\'] else None\n    return profile, raw, qa, runs\n\n\ndef occupied_cells(runs, grid_m):\n    """Coarse equirectangular screening only; never infer exact itinerary equality."""\n    cells = set()\n    for run in runs:\n        for a, b in zip(run, run[1:]):\n            n = max(1, math.ceil(distance(a, b)/(grid_m/2)))\n            for t in np.linspace(0, 1, n+1):\n                lat, lon = a[0]+t*(b[0]-a[0]), a[1]+t*(b[1]-a[1])\n                x = 6371008.8*math.radians(lon+106)*math.cos(math.radians(39))\n                y = 6371008.8*math.radians(lat-39)\n                cells.add((math.floor(x/grid_m), math.floor(y/grid_m)))\n    return cells\n\n\ndef evaluation_groups(rows, cell_sets, threshold):\n    ids = [r[\'route_id\'] for r in rows]\n    parent = {rid: rid for rid in ids}\n    def find(rid):\n        while parent[rid] != rid:\n            parent[rid] = parent[parent[rid]]\n            rid = parent[rid]\n        return rid\n    def union(a, b):\n        parent[find(b)] = find(a)\n    for row in rows:\n        for other in row.get(\'evaluation_group\', \'\').split(\'|\'):\n            if other in parent:\n                union(row[\'route_id\'], other)\n    links = []\n    for i, a in enumerate(ids):\n        for b in ids[i+1:]:\n            size = min(len(cell_sets.get(a, set())), len(cell_sets.get(b, set())))\n            if not size:\n                continue\n            fraction = len(cell_sets[a] & cell_sets[b])/size\n            if fraction >= threshold:\n                union(a, b)\n                links.append({\'a\': a, \'b\': b, \'shorter_track_cell_overlap\': fraction})\n    groups = {}\n    for rid in ids:\n        groups.setdefault(find(rid), []).append(rid)\n    return {rid: \'|\'.join(sorted(groups[find(rid)])) for rid in ids}, links\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)+\'\\n\')\n    tmp.replace(path)\n\n\ndef run_features(project_root, data_root, feature_path, report_path):\n    project_root, data_root = Path(project_root), Path(data_root)\n    artifact_root = project_root/\'outputs\' if (project_root/\'outputs/project_config.json\').exists() else project_root\n    config_path = artifact_root/\'project_config.json\'\n    config = json.loads(config_path.read_text())\n    params = config[\'phase3\'][\'parameters\']\n    if config[\'phase3\'][\'feature_version\'] != FEATURE_VERSION or params[\'threshold_unit\'] != \'percent_grade\':\n        raise ValueError(\'Feature version/unit differs from implementation\')\n    if not (0 <= params[\'medium_grade_min_pct\'] < params[\'extreme_grade_min_pct\'] < params[\'max_abs_grade_pct\']):\n        raise ValueError(\'Invalid grade thresholds\')\n    spec = importlib.util.spec_from_file_location(\'feature_ingestion\', artifact_root/\'block1_scraper.py\')\n    ingestion = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(ingestion)\n    audit_spec = importlib.util.spec_from_file_location(\'feature_audit\', project_root/\'scripts/audit_phase2.py\')\n    audit = importlib.util.module_from_spec(audit_spec)\n    audit_spec.loader.exec_module(audit)\n    snapshot = audit.audit(data_root, project_root)\n    if not snapshot[\'snapshot_integrity_passed\']:\n        raise ValueError(\'Ingestion snapshot failed integrity: \'+str(snapshot[\'errors\']))\n    with (data_root/\'route_status.csv\').open(newline=\'\') as f:\n        statuses = list(csv.DictReader(f))\n    catalog = {x[\'source_url\']: x for x in json.loads((data_root/\'catalog.json\').read_text())}\n    risks = config[\'feature_contract\'][\'risk_encoding\']\n    output, cells, source_hashes = [], {}, {}\n    for status in statuses:\n        rid = status[\'route_id\']\n        path = data_root/\'gpx\'/f\'{rid}.gpx\'\n        sidecar = json.loads(path.with_suffix(\'.json\').read_text())\n        row = {k: status.get(k, \'\') for k in (\'route_id\', \'canonical_name\', \'source_url\', \'manifest_version\',\n                \'evaluation_group\', \'start_name\', \'descent_policy\', \'itinerary_verified\', \'dry_summer_verified\',\n                \'coverage_review_priority\', \'endpoint_gap_m\')}\n        row.update(feature_version=FEATURE_VERSION, feature_scope=\'recorded_source_track\', feature_eligible=False)\n        flags = []\n        source = catalog.get(status[\'source_url\'])\n        reviewed_export = status.get(\'ingestion_method\') == \'approved_local_export\'\n        if reviewed_export:\n            required = (\'reviewer\', \'review_date\', \'source_references\', \'metadata_evidence\',\n                        \'geometry_evidence\', \'conditions_evidence\', \'export_sha256\', \'permission_evidence\')\n            if not all(sidecar.get(k) for k in required):\n                raise ValueError(rid+\': reviewed export lacks scope evidence\')\n            for key in (\'start_policy\', \'summit_order\', \'descent_policy\'):\n                if sidecar.get(key) != status.get(key):\n                    raise ValueError(rid+\': reviewed export itinerary changed\')\n            for key in (\'itinerary_verified\', \'dry_summer_verified\'):\n                if type(sidecar.get(key)) is not bool or sidecar[key] != (status.get(key) == \'true\'):\n                    raise ValueError(rid+\': reviewed export scope flags changed\')\n            source = {k: sidecar[k] for k in (\'yds_raw\', \'yds_class\', \'source_distance_mi\',\n                      \'source_gain_ft\', *(r+\'_raw\' for r in RISKS))}\n            source.update(source_url=status[\'source_url\'], source_html_sha256=\'\',\n                          yds_encoded=config[\'yds_encoding\'][str(source[\'yds_class\'])])\n            row[\'metadata_provenance\'] = \'reviewed_export:\'+sidecar[\'export_sha256\']\n        else:\n            row[\'metadata_provenance\'] = \'captured_official_overview\'\n        if source is None:\n            raise ValueError(rid+\': missing acquired source metadata\')\n        effective = ingestion.apply_selection_metadata(status, source)\n        # Check metadata against the immutable captured overview, rather than trusting CSV edits.\n        for key in (\'yds_raw\', \'yds_class\', \'yds_encoded\'):\n            value = effective.get(key)\n            if value is None:\n                raise ValueError(rid+\': missing class\')\n            if key == \'yds_raw\':\n                if status.get(key) != str(value):\n                    raise ValueError(rid+\': metadata mismatch \'+key)\n            elif float(status[key]) != value:\n                raise ValueError(rid+\': encoding mismatch \'+key)\n            row[key] = value\n        row[\'source_html_sha256\'] = source[\'source_html_sha256\']\n        for risk in RISKS:\n            label = status[risk+\'_raw\']\n            if label != source[risk+\'_raw\'] or label not in risks:\n                raise ValueError(rid+\': unknown or altered source risk \'+risk)\n            row[risk+\'_raw\'], row[risk+\'_encoded\'] = label, risks[label]\n        for key in (\'source_distance_mi\', \'source_gain_ft\'):\n            value = effective.get(key)\n            supplied = float(status[key]) if status.get(key) else None\n            if supplied != value:\n                raise ValueError(rid+\': published metric changed \'+key)\n            row[key] = value\n        if status.get(\'metadata_complete\', \'\').lower() != \'true\':\n            flags.append(\'metadata_scope_pending\')\n        if status.get(\'itinerary_verified\') != \'true\' or status.get(\'dry_summer_verified\') != \'true\':\n            flags.append(\'complete_dry_itinerary_unverified\')\n        body = path.read_bytes()\n        quality = ingestion.validate_gpx(body)\n        if not reviewed_export and sidecar.get(\'source_html_sha256\') != source[\'source_html_sha256\']:\n            raise ValueError(rid+\': overview provenance mismatch\')\n        receipt = sidecar.get(\'acquisition_receipt\')\n        if receipt and receipt.get(\'metadata_sha256\') != source[\'source_html_sha256\']:\n            raise ValueError(rid+\': acquisition overview hash mismatch\')\n        row[\'gpx_sha256\'] = quality[\'gpx_sha256\']\n        row[\'geometry_sha256\'] = quality[\'geometry_sha256\']\n        source_hashes[rid] = {\'gpx\': row[\'gpx_sha256\'], \'overview\': row[\'source_html_sha256\'],\n                              \'reviewed_export\': sidecar.get(\'export_sha256\', \'\')}\n        if status.get(\'itinerary_verified\') == \'true\' and not all(sidecar.get(k) for k in\n                (\'reviewer\', \'review_date\', \'geometry_evidence\', \'conditions_evidence\')):\n            flags.append(\'scope_review_evidence_missing\')\n        profile, raw, qa, runs = analyze_track(sequences(body), params)\n        cells[rid] = occupied_cells(runs, params[\'overlap_grid_m\'])\n        row.update({\'source_track_\'+k: v for k, v in profile.items()})\n        row.update({\'raw_track_\'+k: v for k, v in raw.items()})\n        row.update({\'profile_\'+k: v for k, v in qa.items()})\n        row.update(accepted_segment_densities(profile, qa))\n        if qa[\'valid_distance_fraction\'] < params[\'min_valid_distance_fraction\']:\n            flags.append(\'insufficient_valid_profile_coverage\')\n        if any(qa[k] for k in (\'long_gap\', \'missing_elevation\', \'implausible_grade\')):\n            flags.append(\'profile_edges_excluded\')\n        if profile[\'distance_mi\'] is None:\n            flags.append(\'no_valid_profile\')\n        if any(qa[k] is not None and qa[k] > params[\'max_density_sensitivity_pp\']\n               for k in (\'density_medium_range_pp\', \'density_extreme_range_pp\')):\n            flags.append(\'density_sensitive_to_parameters\')\n        if qa[\'gain_sensitivity_pct\'] is not None and qa[\'gain_sensitivity_pct\'] > params[\'max_gain_sensitivity_pct\']:\n            flags.append(\'gain_sensitive_to_parameters\')\n        if qa[\'valid_distance_fraction\'] < params[\'min_valid_distance_fraction\']:\n            # Keep the miles of accepted observations, but do not claim route-wide densities.\n            for key in (\'crux_density_medium\', \'crux_density_extreme\'):\n                row[\'source_track_\'+key] = None\n        for feature, source_key in [(\'distance_mi\', \'source_distance_mi\'), (\'gain_ft\', \'source_gain_ft\')]:\n            total = row[source_key]\n            measured = profile[feature]\n            ratio = measured/total if total and measured is not None else None\n            row[\'recorded_to_published_\'+feature+\'_ratio\'] = ratio\n            # A mismatch on an unreviewed ascent file is diagnostic, not a false full-route test.\n            if status.get(\'itinerary_verified\') == \'true\' and ratio is not None:\n                limit = params[\'distance_discrepancy_fraction\' if feature == \'distance_mi\' else \'gain_discrepancy_fraction\']\n                if abs(ratio-1) > limit:\n                    flags.append(feature+\'_scope_discrepancy\')\n        row[\'profile_qa_pass\'] = not any(x not in (\'complete_dry_itinerary_unverified\', \'metadata_scope_pending\') for x in flags)\n        row[\'feature_eligible\'] = status.get(\'training_eligible\', \'\').lower() == \'true\' and row[\'profile_qa_pass\']\n        for key in GEOMETRY_FEATURES:\n            row[key] = profile[key] if row[\'feature_eligible\'] else None\n        row[\'feature_scope\'] = \'complete_reviewed_itinerary\' if row[\'feature_eligible\'] else \'recorded_source_track\'\n        row[\'qa_flags\'] = \'|\'.join(flags)\n        output.append(row)\n    groups, links = evaluation_groups(output, cells, params[\'overlap_fraction_threshold\'])\n    for row in output:\n        row[\'evaluation_group\'] = groups[row[\'route_id\']]\n    feature_path = Path(feature_path)\n    feature_path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = feature_path.with_suffix(\'.csv.tmp\')\n    fields = list(dict.fromkeys(k for row in output for k in row))\n    with tmp.open(\'w\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fields, lineterminator=\'\\n\')\n        writer.writeheader()\n        writer.writerows(output)\n    tmp.replace(feature_path)\n    flags = Counter(flag for row in output for flag in row[\'qa_flags\'].split(\'|\') if flag)\n    eligible = sum(row[\'feature_eligible\'] for row in output)\n    review = {\'feature_version\': FEATURE_VERSION, \'manifest_version\': config[\'manifest_version\'],\n              \'manifest_sha256\': config[\'manifest_sha256\'], \'config_sha256\': digest(config_path),\n              \'implementation_sha256\': digest(artifact_root/\'block2_features.py\'),\n              \'source_provenance_sha256\': hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest(),\n              \'input_catalog_sha256\': digest(data_root/\'catalog.json\'),\n              \'input_status_sha256\': digest(data_root/\'route_status.csv\'),\n              \'feature_table_sha256\': digest(feature_path), \'run_at\': datetime.now(timezone.utc).isoformat(),\n              \'runtime\': {\'numpy\': np.__version__, \'python\': ingestion.sys.version.split()[0]},\n              \'snapshot_integrity_passed\': snapshot[\'snapshot_integrity_passed\'],\n              \'parameters\': params, \'route_count\': len(output), \'recorded_profiles_computed\': len(output),\n              \'class_and_risk_encodings_complete\': sum(all(row.get(k) is not None for k in\n                  (\'yds_encoded\', *(r+\'_encoded\' for r in RISKS))) for row in output),\n              \'profile_qa_pass_count\': sum(row[\'profile_qa_pass\'] for row in output),\n              \'feature_eligible_count\': eligible, \'phase3_engineering_complete\': True,\n              \'phase3_dataset_ready\': eligible == len(output) == 100,\n              \'phase4_engineering_ready\': True, \'phase4_complete_itinerary_pca_ready\': eligible == len(output) == 100,\n              \'qa_flag_counts\': dict(flags), \'approximate_overlap_link_count\': len(links),\n              \'evaluation_groups\': sorted(set(groups.values())),\n              \'overlap_screening_note\': \'50m grid common-cell fraction of shorter recorded path; coarse candidates, not exact duplicates or verified full-itinerary overlaps.\',\n              \'remaining_work\': [\'Review complete dry start/branch/summit/descent coverage and document assemblies.\',\n                                 \'Resolve Culebra gain units and North Eolus complete-itinerary totals.\',\n                                 \'Review excluded edges and sensitivity before freezing feature parameters.\',\n                                 \'Review approximate spatial-overlap groups before model evaluation.\']}\n    atomic_json(report_path, review)\n    return review\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    root = Path(__file__).resolve().parents[1]\n    parser.add_argument(\'--project-root\', type=Path, default=root)\n    parser.add_argument(\'--data-root\', type=Path, default=root/\'private_data/raw\')\n    parser.add_argument(\'--features\', type=Path, default=root/\'private_data/routes_features.csv\')\n    parser.add_argument(\'--report\', type=Path, default=root/\'outputs/phase3_review.json\')\n    args = parser.parse_args()\n    result = run_features(args.project_root, args.data_root, args.features, args.report)\n    print(json.dumps({k: result[k] for k in (\'route_count\', \'recorded_profiles_computed\',\n          \'class_and_risk_encodings_complete\', \'profile_qa_pass_count\', \'feature_eligible_count\',\n          \'phase3_dataset_ready\', \'qa_flag_counts\', \'approximate_overlap_link_count\')}, indent=2))\n'


In [ ]:
FEATURE_PROJECT_ROOT = Path('/content/colorado_14er_kickoff')  # Extracted current ZIP; edit if needed.
FEATURE_ARTIFACT_ROOT = FEATURE_PROJECT_ROOT / 'outputs' if (FEATURE_PROJECT_ROOT / 'outputs' / 'project_config.json').exists() else FEATURE_PROJECT_ROOT
if not (FEATURE_ARTIFACT_ROOT / 'project_config.json').exists():
    raise FileNotFoundError('Extract the current kickoff ZIP and set FEATURE_PROJECT_ROOT.')

feature_namespace = {'__name__': 'phase3_notebook'}
exec(compile(FEATURE_SOURCE, 'block2_features.py', 'exec'), feature_namespace)
if feature_namespace['digest'](FEATURE_ARTIFACT_ROOT / 'block2_features.py') != hashlib.sha256(FEATURE_SOURCE.encode()).hexdigest():
    raise ValueError('Notebook and extracted Block 2 differ; use the same synchronized package.')
feature_review = feature_namespace['run_features'](
    FEATURE_PROJECT_ROOT,
    Path(DATA_ROOT) / 'raw',
    Path(DATA_ROOT) / 'routes_features.csv',
    Path(DATA_ROOT) / 'phase3_review.json',
)
print(json.dumps({k: feature_review[k] for k in ('route_count', 'recorded_profiles_computed',
      'profile_qa_pass_count', 'feature_eligible_count', 'phase3_dataset_ready')}, indent=2))
display(pd.read_csv(Path(DATA_ROOT) / 'routes_features.csv', keep_default_na=False)[[
    'route_id', 'canonical_name', 'source_track_crux_density_medium',
    'source_track_crux_density_extreme', 'source_track_steep_above20_mi',
    'source_track_longest_extreme_mi', 'feature_eligible', 'qa_flags']])
